# Intra-View Redundancy Analysis for BGP Graph Features

This notebook performs comprehensive **intra-view redundancy analysis** on the extracted BGP graph features
(graph-level, node-level, and relationship-level) to identify and remove redundant features within each view.

## Methods Applied

| # | Method | What it captures | Output |
|---|--------|-----------------|--------|
| 1 | **Distance Correlation (dCor)** | Non-linear + linear dependencies | Pairwise matrix |
| 2 | **Spearman Rank Correlation** | Monotonic relationships | Pairwise matrix |
| 3 | **Pearson Correlation** | Linear relationships | Pairwise matrix |
| 4 | **Variance Inflation Factor (VIF)** | Multicollinearity | Per-feature score |
| 5 | **Hierarchical Clustering** | Collinear feature groups | Dendrogram + clusters |
| 6 | **mRMR (Min Redundancy Max Relevance)** | Relevance vs. redundancy trade-off | Ranked feature list |
| 7 | **Mutual Information** | General statistical dependence | Pairwise matrix |
| 8 | **Condition Number Analysis** | Numerical stability / collinearity | Scalar per view |

Each method is applied **separately to each feature view** (graph-level, node-level, relationship-level).

## 1. Installation & Imports

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install dcor scikit-learn scipy numpy pandas matplotlib seaborn statsmodels mrmr-selection

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# Correlation methods
from scipy import stats as sp_stats
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
import dcor  # Distance correlation

# Multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# Mutual Information
from sklearn.feature_selection import mutual_info_regression

# mRMR
try:
    from mrmr import mrmr_regression
    HAS_MRMR = True
except ImportError:
    HAS_MRMR = False
    print("WARNING: mrmr-selection not installed. Install with: pip install mrmr-selection")

warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams.update({
    'figure.figsize': (14, 10),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print("All imports successful.")
print(f"  dcor version: {dcor.__version__}")
print(f"  mRMR available: {HAS_MRMR}")

## 2. Configuration & Data Loading

In [ ]:

# ============================================================================
# CONFIGURATION
# ============================================================================

# ── MODE SELECTION ──
# Options: 'graph', 'statistical', 'combined'
#   'graph'        -> legacy compatibility mode for original graph-only CSVs
#   'statistical'  -> legacy compatibility mode for original statistical-only CSVs
#   'combined'     -> canonical mode using the unified aligned CSV
MODE = 'combined'  # <── CHANGE THIS TO SWITCH MODES

# ── Common settings ──
COLLECTOR = "rrc05"
TARGET_AS = 12880
EGO_K_HOP = 2
START_DATE = "2025-11-01"
END_DATE = "2025-11-30"
WINDOW_SIZE = "5min"

# ── Optional direct-path overrides (recommended for site/job outputs) ──
# In combined mode this can point to either the raw unified feature CSV
# or the *_discovered.csv output produced by the labeling notebook.
UNIFIED_FULL_PATH_OVERRIDE = Path(
    "/home/smotaali/First_Full_Paper/dataset/phase1_labels_hdbscan/"
    "unified_full_rrc05_AS12880_2hop_2025-11-01_2025-11-30_5min_discovered.csv"
)
ANALYSIS_DIR_OVERRIDE = None

# ── Label-aware filtering for combined mode ──
# Options: 'all', 'likely_normal', 'uncertain', 'likely_anomaly',
#          'high_confidence_anomaly', 'anomalies', 'non_normal'
LABEL_COLUMN = 'discovered_label'
LABEL_FILTER = 'anomalies'
ANALYSIS_DATASET_LABEL = None  # None -> auto-derived from LABEL_FILTER

def _slugify_token(value):
    text = str(value).strip().lower()
    slug = ''.join(ch if ch.isalnum() else '_' for ch in text)
    slug = '_'.join(part for part in slug.split('_') if part)
    return slug or 'dataset'

def _resolve_analysis_dataset_label(dataset_label, label_filter):
    if dataset_label:
        return _slugify_token(dataset_label)
    if label_filter == 'all':
        return 'full'
    return _slugify_token(label_filter)

ANALYSIS_DATASET_LABEL = _resolve_analysis_dataset_label(
    ANALYSIS_DATASET_LABEL,
    LABEL_FILTER,
)

def _normalize_label_filter_tokens(label_filter):
    if isinstance(label_filter, str):
        values = [label_filter]
    elif isinstance(label_filter, (list, tuple, set)):
        values = list(label_filter)
    else:
        values = [label_filter]
    return {_slugify_token(v) for v in values if v is not None}

def _resolve_daily_stability_setting(explicit_setting, label_filter):
    if explicit_setting is not None:
        return bool(explicit_setting), 'manual override'

    normalized = _normalize_label_filter_tokens(label_filter)
    daily_ok_filters = {'all', 'full', 'likely_normal'}

    if normalized and normalized.issubset(daily_ok_filters):
        return True, 'auto-enabled for full/likely-normal analysis'

    return False, 'auto-disabled for subset/anomaly-focused analysis'

# ── Daily analysis configuration (combined mode, applied after label filtering) ──
# Set to None to let the notebook decide automatically from LABEL_FILTER.
# Use True/False only if you want to override the auto policy.
ANALYZE_DAILY_STABILITY = None
ANALYZE_DAILY_STABILITY, ANALYZE_DAILY_STABILITY_REASON = _resolve_daily_stability_setting(
    ANALYZE_DAILY_STABILITY,
    LABEL_FILTER,
)
DAILY_PRIMARY_METHOD = 'spearman'  # method used for labels and headline plots
DAILY_PAIRWISE_METHODS = ['pearson', 'spearman', 'dcor']
DAILY_ENABLE_MI = False  # heavier; set True if you want daily MI matrices too
DAILY_ENABLE_VIF = True
DAILY_ENABLE_CONDITION = True
DAILY_DCOR_MAX_SAMPLES = 192  # subsample daily dCor for tractable runtime
DAILY_MI_MAX_SAMPLES = 192
DAILY_CV_MIN_MEAN_ABS = 1e-4
DAILY_EXPORT_PAIRWISE_MATRICES = True
DAILY_EXPORT_HEATMAPS = True
DAILY_EXPORT_METHOD_COMPARISON = True
DAILY_HEATMAP_TOP_FEATURES = 24
DAILY_HEATMAP_BALANCE_DOMAINS = True
DAILY_HEATMAP_MAX_DAYS = None  # set an int if you want to limit PNG generation
DAILY_MIN_ROWS = 144  # at least half a day of 5-minute windows
STABLE_PAIR_ABS_CORR_THRESHOLD = 0.90
STABLE_PAIR_MIN_DAY_FRACTION = 0.80
STABLE_PAIR_MAX_STD = 0.10
TOP_DAILY_PAIR_COUNT = 25

if DAILY_ENABLE_MI and 'mi' not in DAILY_PAIRWISE_METHODS:
    DAILY_PAIRWISE_METHODS.append('mi')
if DAILY_PRIMARY_METHOD not in DAILY_PAIRWISE_METHODS:
    raise ValueError(
        f"DAILY_PRIMARY_METHOD must be in DAILY_PAIRWISE_METHODS. "
        f"Got {DAILY_PRIMARY_METHOD!r} vs {DAILY_PAIRWISE_METHODS!r}"
    )

# ── Paths per mode ──
GRAPH_BASE_DIR = Path("/home/smotaali/First_Full_Paper/bgp_graph_features_results")
STAT_BASE_DIR = Path("/home/smotaali/First_Full_Paper/bgp_stat_features_results")
UNIFIED_BASE_DIR = Path("/home/smotaali/First_Full_Paper/bgp_unified_results")

graph_suffix = f"{COLLECTOR}_AS{TARGET_AS}_{EGO_K_HOP}hop_{START_DATE}_{END_DATE}"
stat_suffix = f"{COLLECTOR}_AS{TARGET_AS}_{EGO_K_HOP}hop_{START_DATE}_{END_DATE}"
unified_suffix = f"{COLLECTOR}_AS{TARGET_AS}_{EGO_K_HOP}hop_{START_DATE}_{END_DATE}_{WINDOW_SIZE}"

# Graph feature paths (for MODE='graph')
GRAPH_OUTPUT_DIR = GRAPH_BASE_DIR / "output"
GRAPH_TS_PATH = GRAPH_OUTPUT_DIR / f"graph_level_timeseries_{graph_suffix}.csv"
NODE_TS_PATH = GRAPH_OUTPUT_DIR / f"node_level_timeseries_{graph_suffix}.csv"

# Statistical feature paths (for MODE='statistical')
STAT_OUTPUT_DIR = STAT_BASE_DIR / "output"
STAT_TS_PATH = STAT_OUTPUT_DIR / f"stat_features_full_{stat_suffix}.csv"

# Unified feature path (for MODE='combined')
UNIFIED_OUTPUT_DIR = UNIFIED_BASE_DIR / "output" / f"{COLLECTOR}_AS{TARGET_AS}_{EGO_K_HOP}hop_{START_DATE}_{END_DATE}"
_DEFAULT_UNIFIED_FULL_PATH = UNIFIED_OUTPUT_DIR / f"unified_full_{unified_suffix}.csv"
UNIFIED_FULL_PATH = Path(UNIFIED_FULL_PATH_OVERRIDE) if UNIFIED_FULL_PATH_OVERRIDE else _DEFAULT_UNIFIED_FULL_PATH
COMBINED_OUTPUT_ROOT = UNIFIED_FULL_PATH.parent if UNIFIED_FULL_PATH_OVERRIDE else UNIFIED_OUTPUT_DIR

# ── Analysis output directory (per mode) ──
if ANALYSIS_DIR_OVERRIDE:
    ANALYSIS_DIR = Path(ANALYSIS_DIR_OVERRIDE)
elif MODE == 'graph':
    ANALYSIS_DIR = GRAPH_BASE_DIR / f"redundancy_analysis/{graph_suffix}"
elif MODE == 'statistical':
    ANALYSIS_DIR = STAT_BASE_DIR / f"redundancy_analysis/{stat_suffix}"
elif MODE == 'combined':
    ANALYSIS_DIR = COMBINED_OUTPUT_ROOT / f"correlation_analysis_{UNIFIED_FULL_PATH.stem}_{ANALYSIS_DATASET_LABEL}"
else:
    raise ValueError(f"Unknown MODE: {MODE}. Use 'graph', 'statistical', or 'combined'.")

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# ── Thresholds ──
CORRELATION_THRESHOLD = 0.90
VIF_THRESHOLD = 10.0
CLUSTER_DISTANCE_THRESHOLD = 0.3
MIN_ROWS_FOR_DCOR = 3
MIN_ROWS_FOR_CORRELATION = 3
MIN_ROWS_FOR_MI = 3
MIN_ROWS_FOR_CONDITION = 3
MIN_ROWS_FOR_VIF = 8
MIN_NODE_SNAPSHOT_COVERAGE = 0.75

print(f"{'='*60}")
print(f"MODE: {MODE}")
print(f"{'='*60}")
if MODE == 'graph':
    print(f"Graph-level CSV: {GRAPH_TS_PATH}")
    print(f"Node-level CSV:  {NODE_TS_PATH}")
elif MODE == 'statistical':
    print(f"Statistical CSV: {STAT_TS_PATH}")
elif MODE == 'combined':
    print(f"Unified input CSV: {UNIFIED_FULL_PATH}")
    print(f"Combined analysis dataset label: {ANALYSIS_DATASET_LABEL}")
    print(f"Label column: {LABEL_COLUMN}")
    print(f"Label filter: {LABEL_FILTER}")
    print(f"Combined analysis save root: {COMBINED_OUTPUT_ROOT}")
    print(f"Daily suite enabled: {ANALYZE_DAILY_STABILITY} ({ANALYZE_DAILY_STABILITY_REASON})")
    print(f"Daily pairwise methods: {DAILY_PAIRWISE_METHODS}")
    print(f"Daily primary method: {DAILY_PRIMARY_METHOD}")
    print(f"Daily heatmaps enabled: {DAILY_EXPORT_HEATMAPS}")
    print(f"Daily method comparison enabled: {DAILY_EXPORT_METHOD_COMPARISON}")
    print(f"Daily dCor max samples: {DAILY_DCOR_MAX_SAMPLES}")
    if DAILY_EXPORT_HEATMAPS:
        print(f"Daily heatmap top features: {DAILY_HEATMAP_TOP_FEATURES}")
print(f"Analysis output: {ANALYSIS_DIR}")


In [ ]:

# ============================================================================
# Load Data (mode-dependent)
# ============================================================================

if MODE == 'graph':
    graph_ts_df = pd.read_csv(GRAPH_TS_PATH)
    node_ts_df = pd.read_csv(NODE_TS_PATH)
    print(f"Graph-level time series: {graph_ts_df.shape}")
    print(f"Node-level time series:  {node_ts_df.shape}")
    print(f"  Unique snapshots: {node_ts_df['snapshot_id'].nunique()}")
    print(f"  Unique ASes: {node_ts_df['asn'].nunique()}")

elif MODE == 'statistical':
    stat_ts_df = pd.read_csv(STAT_TS_PATH)
    stat_ts_df['window_start'] = pd.to_datetime(stat_ts_df['window_start'])
    print(f"Statistical time series: {stat_ts_df.shape}")
    print(f"  Windows: {len(stat_ts_df)}")
    print(f"  Columns: {list(stat_ts_df.columns)}")

elif MODE == 'combined':
    if not UNIFIED_FULL_PATH.exists():
        raise FileNotFoundError(f"Unified input CSV not found: {UNIFIED_FULL_PATH}")

    unified_source_df = pd.read_csv(UNIFIED_FULL_PATH)
    if 'window_start' in unified_source_df.columns:
        unified_source_df['window_start'] = pd.to_datetime(unified_source_df['window_start'])

    print(f"Unified input dataset: {unified_source_df.shape}")
    print(f"  Rows before filtering: {len(unified_source_df)}")

    if LABEL_COLUMN in unified_source_df.columns:
        print(f"  Label distribution in input ({LABEL_COLUMN}):")
        print(unified_source_df[LABEL_COLUMN].fillna('missing').value_counts(dropna=False).to_string())
    elif LABEL_FILTER != 'all':
        raise KeyError(
            f"Requested LABEL_FILTER={LABEL_FILTER!r}, but column {LABEL_COLUMN!r} is missing from {UNIFIED_FULL_PATH}"
        )

    if LABEL_FILTER == 'all':
        unified_full_df = unified_source_df.copy()
        label_filter_desc = 'all rows'
    elif LABEL_FILTER == 'anomalies':
        keep_labels = ['likely_anomaly', 'high_confidence_anomaly']
        unified_full_df = unified_source_df[unified_source_df[LABEL_COLUMN].isin(keep_labels)].copy()
        label_filter_desc = f"{LABEL_COLUMN} in {keep_labels}"
    elif LABEL_FILTER == 'non_normal':
        keep_labels = ['uncertain', 'likely_anomaly', 'high_confidence_anomaly']
        unified_full_df = unified_source_df[unified_source_df[LABEL_COLUMN].isin(keep_labels)].copy()
        label_filter_desc = f"{LABEL_COLUMN} in {keep_labels}"
    else:
        keep_labels = [LABEL_FILTER]
        unified_full_df = unified_source_df[unified_source_df[LABEL_COLUMN].isin(keep_labels)].copy()
        label_filter_desc = f"{LABEL_COLUMN} == {LABEL_FILTER!r}"

    if len(unified_full_df) == 0:
        raise ValueError(
            f"Label filter {label_filter_desc} produced 0 rows. Choose a different LABEL_FILTER or input CSV."
        )

    combined_non_feature_cols = {
        'window_start', 'window_end', 'window_id', 'collector', 'asn', 'segment',
        'incident', 'label', 'label_rule', 'label_refined', 'label_discovered', 'discovered_label',
        'cluster', 'hdbscan_cluster', 'kmeans_cluster',
        'iso_forest_score', 'lof_score', 'statistical_score', 'elliptic_score', 'hdbscan_outlier_score',
        'hdbscan_unscored', 'equal_consensus_score', 'anomaly_votes', 'consensus_score',
        'discovered_label_pre_kmeans', 'kmeans_small_cluster_flag', 'kmeans_label_boosted',
        'label_stability_score', 'label_flip_rate', 'bootstrap_modal_label', 'bootstrap_modal_agreement',
        'fourier_template', 'fourier_residual', 'fourier_residual_z'
    }

    print(f"Filtered analysis dataset: {unified_full_df.shape}")
    print(f"  Applied row filter: {label_filter_desc}")
    print(f"  Feature columns: {len([c for c in unified_full_df.columns if c not in combined_non_feature_cols])}")
    if 'window_start' in unified_full_df.columns:
        print(f"  Time range: {unified_full_df['window_start'].min()} to {unified_full_df['window_start'].max()}")
        print(f"  Days: {unified_full_df['window_start'].dt.date.nunique()}")


In [ ]:

# ============================================================================
# Separate feature views (mode-dependent)
# ============================================================================

# Common meta columns
META_COLS = ['snapshot_id', 'timestamp', 'collector']
NODE_META_COLS = ['snapshot_id', 'timestamp', 'collector', 'asn']
STAT_META_COLS = ['window_start', 'window_end', 'collector', 'window_size']
UNIFIED_META_COLS = [
    'window_start', 'window_end', 'window_id', 'collector', 'asn', 'segment',
    'incident', 'label', 'label_rule', 'label_refined', 'label_discovered', 'discovered_label',
    'cluster', 'hdbscan_cluster', 'kmeans_cluster',
    'iso_forest_score', 'lof_score', 'statistical_score', 'elliptic_score', 'hdbscan_outlier_score',
    'hdbscan_unscored', 'equal_consensus_score', 'anomaly_votes', 'consensus_score',
    'discovered_label_pre_kmeans', 'kmeans_small_cluster_flag', 'kmeans_label_boosted',
    'label_stability_score', 'label_flip_rate', 'bootstrap_modal_label', 'bootstrap_modal_agreement',
    'fourier_template', 'fourier_residual', 'fourier_residual_z'
]
FLAG_COLS = ['diameter_approximate', 'symmetry_ratio_partial', 'natural_connectivity_partial']

# ── Define feature groups ──
RELATIONSHIP_GRAPH_FEATURES = [
    'rel_coverage', 'frac_p2c_edges', 'frac_p2p_edges', 'frac_unknown_edges',
    'n_tier1_in_subgraph', 'n_ixp_in_subgraph', 'valley_free_violations',
    'valley_free_violation_rate', 'valley_free_paths_checked',
    'avg_violation_depth', 'max_violation_depth'
]

RELATIONSHIP_NODE_FEATURES = [
    'n_providers', 'n_customers', 'n_peers', 'n_unknown_rel',
    'p2c_ratio', 'p2p_ratio', 'provider_ratio', 'is_tier1', 'is_ixp'
]

# Statistical feature category definitions
STAT_VOLUME_FEATURES = [
    'announcements', 'withdrawals', 'total_updates',
    'ann_rate', 'wd_rate', 'wd_ann_ratio', 'ann_wd_ratio'
]
STAT_PREFIX_FEATURES = [
    'unique_prefixes_ann', 'unique_prefixes_wd', 'new_prefixes', 'dups', 'flaps'
]
STAT_ORIGIN_FEATURES = ['origin_IGP', 'origin_INCOMPLETE', 'origin_changes']
STAT_PATH_FEATURES = [
    'as_path_avg', 'as_path_max', 'as_path_std', 'unique_as_path_max',
    'edit_distance_avg', 'edit_distance_max',
    'edit_distance_dict_0', 'edit_distance_dict_1', 'edit_distance_dict_2',
    'edit_distance_dict_3', 'edit_distance_dict_4', 'edit_distance_dict_5',
    'edit_distance_dict_6', 'edit_distance_unique_dict_0', 'edit_distance_unique_dict_1'
]
STAT_RARE_FEATURES = ['number_rare_ases', 'rare_ases_ratio']
STAT_IMPWD_FEATURES = ['imp_wd', 'imp_wd_spath', 'imp_wd_dpath']
STAT_BEHAVIORAL_FEATURES = ['nadas', 'unique_peers']

ALL_STAT_FEATURES = (STAT_VOLUME_FEATURES + STAT_PREFIX_FEATURES + STAT_ORIGIN_FEATURES +
                     STAT_PATH_FEATURES + STAT_RARE_FEATURES + STAT_IMPWD_FEATURES +
                     STAT_BEHAVIORAL_FEATURES)

# ── FULL feature lists (all non-meta features from the aligned unified CSV) ──
FULL_GRAPH_LEVEL = [
    'n_nodes', 'n_edges', 'density', 'diameter', 'radius', 'avg_path_length',
    'assortativity', 'clustering_global', 'clustering_avg_local',
    'edge_connectivity', 'node_connectivity',
    'betweenness_mean', 'betweenness_std', 'betweenness_max', 'betweenness_skewness',
    'core_mean', 'core_median', 'core_std', 'degeneracy', 'innermost_core_size',
    'rich_club_p25', 'rich_club_p50', 'rich_club_p75', 'rich_club_p90', 'rich_club_p95',
    'algebraic_connectivity', 'spectral_gap', 'spectral_radius', 'adj_eig_ratio_1_2',
    'natural_connectivity', 'symmetry_ratio', 'percolation_limit',
    'kirchhoff_index', 'log_spanning_trees',
    'avg_edge_ixp_cosine_dist', 'std_edge_ixp_cosine_dist', 'median_edge_ixp_cosine_dist',
    'avg_ixp_cosine_dist', 'ixp_vector_norm',
    'n_edges_with_ixp_data', 'n_edges_missing_ixp_data', 'n_ixp_memberships',
    'mean_jaccard', 'mean_adamic_adar',
    'rel_coverage', 'frac_p2c_edges', 'frac_p2p_edges', 'frac_unknown_edges',
    'n_tier1_in_subgraph', 'n_ixp_in_subgraph',
    'valley_free_violations', 'valley_free_violation_rate', 'valley_free_paths_checked',
    'avg_violation_depth', 'max_violation_depth', 'vf_rate_delta',
    'is_tier1', 'is_ixp', 'n_providers', 'n_customers', 'n_peers', 'n_unknown_rel',
    'p2c_ratio', 'p2p_ratio', 'provider_ratio',
    'ego_nodes', 'ego_nodes_dynamic', 'ego_edges', 'ego_valley_free_violations',
    'ego_valley_free_violation_rate', 'ego_valley_free_paths_checked',
    'ego_valley_free_paths_total', 'ego_vf_paths_frac',
    'ego_avg_violation_depth', 'ego_max_violation_depth',
    'ego_transit_violations', 'ego_transit_vf_rate', 'ego_origin_violations',
]
FULL_NODE_LEVEL = [
    'degree', 'degree_centrality', 'betweenness_centrality', 'closeness_centrality',
    'eigenvector_centrality', 'pagerank', 'core_number', 'eccentricity',
    'local_clustering', 'avg_neighbor_degree', 'node_clique_number',
]
FULL_STATISTICAL = [
    'announcements', 'withdrawals', 'total_updates', 'ann_rate', 'wd_rate',
    'wd_ann_ratio', 'ann_wd_ratio',
    'unique_prefixes_ann', 'unique_prefixes_wd', 'new_prefixes', 'dups', 'flaps',
    'origin_IGP', 'origin_INCOMPLETE', 'origin_changes',
    'as_path_avg', 'as_path_max', 'as_path_std', 'unique_as_path_max',
    'edit_distance_avg', 'edit_distance_max',
    'edit_distance_dict_0', 'edit_distance_dict_1', 'edit_distance_dict_2',
    'edit_distance_dict_3', 'edit_distance_dict_4', 'edit_distance_dict_5',
    'edit_distance_dict_6', 'edit_distance_unique_dict_0', 'edit_distance_unique_dict_1',
    'number_rare_ases', 'rare_ases_ratio',
    'imp_wd', 'imp_wd_spath', 'imp_wd_dpath',
    'nadas', 'unique_peers', 'ego_filter_ratio',
    'ego_updates_total', 'ego_updates_filtered',
]

views = {}

if MODE == 'graph':
    views = {
        'graph_level': (graph_ts_df, [c for c in graph_ts_df.columns if c not in META_COLS]),
        'node_level': (node_ts_df, [c for c in node_ts_df.columns if c not in NODE_META_COLS]),
    }
elif MODE == 'statistical':
    views = {
        'statistical': (stat_ts_df, [c for c in stat_ts_df.columns if c not in STAT_META_COLS]),
    }
elif MODE == 'combined':
    def _build_combined_views(udf, dataset_label):
        prefix = _slugify_token(dataset_label)
        v = {}
        all_feats = [c for c in udf.columns if c not in UNIFIED_META_COLS]
        v[f'{prefix}_cross_view_ALL'] = (udf, all_feats)

        avail_graph = [c for c in FULL_GRAPH_LEVEL if c in udf.columns]
        if avail_graph:
            v[f'{prefix}_cross_graph_level'] = (udf, avail_graph)

        avail_node = [c for c in FULL_NODE_LEVEL if c in udf.columns]
        if avail_node:
            v[f'{prefix}_cross_node_level'] = (udf, avail_node)

        avail_stat = [c for c in FULL_STATISTICAL if c in udf.columns]
        if avail_stat:
            v[f'{prefix}_cross_statistical'] = (udf, avail_stat)

        cross_feats = avail_graph + avail_node + avail_stat
        if cross_feats:
            v[f'{prefix}_cross_graph_vs_stat'] = (udf, cross_feats)

        return v, len(all_feats), len(avail_graph), len(avail_node), len(avail_stat)

    combined_views, full_all, full_g, full_n, full_s = _build_combined_views(
        unified_full_df,
        ANALYSIS_DATASET_LABEL,
    )
    views.update(combined_views)
    print(f"Combined-mode features available after filtering: {full_all}")
    print(f"  Graph-level: {full_g}")
    print(f"  Node-level:  {full_n}")
    print(f"  Statistical: {full_s}")
    print(f"  Active dataset label: {ANALYSIS_DATASET_LABEL}")

print(f"\n{'='*60}")
print(f"Feature Views for MODE='{MODE}':")
print(f"{'='*60}")
for name, (df, cols) in views.items():
    avail = [c for c in cols if c in df.columns]
    print(f"  {name}: {len(avail)} features, {len(df)} rows")


In [ ]:
# ============================================================================
# Prepare clean numeric DataFrames for each view
# ============================================================================

def prepare_view(df, feature_cols, view_name):
    """
    Extract numeric features, drop constant/all-NaN columns, impute remaining NaNs.
    Returns cleaned DataFrame and list of dropped columns.
    """
    available = [c for c in feature_cols if c in df.columns]
    view_df = df[available].copy()

    # Convert to numeric, coerce errors
    view_df = view_df.apply(pd.to_numeric, errors='coerce')

    # Drop columns that are all NaN
    all_nan = view_df.columns[view_df.isna().all()].tolist()
    if all_nan:
        print(f"  [{view_name}] Dropping all-NaN columns: {all_nan}")
        view_df = view_df.drop(columns=all_nan)

    # Drop constant columns (zero variance)
    constant = view_df.columns[view_df.nunique() <= 1].tolist()
    if constant:
        print(f"  [{view_name}] Dropping constant columns: {constant}")
        view_df = view_df.drop(columns=constant)

    # Report NaN summary
    nan_pct = view_df.isna().mean()
    cols_with_nan = nan_pct[nan_pct > 0]
    if len(cols_with_nan) > 0:
        print(f"  [{view_name}] Columns with NaN (will be median-imputed):")
        for col, pct in cols_with_nan.items():
            print(f"    {col}: {pct:.1%} missing")

    # Impute NaNs with column median
    view_df = view_df.fillna(view_df.median())

    dropped = all_nan + constant
    print(f"  [{view_name}] Final shape: {view_df.shape} "
          f"(dropped {len(dropped)} columns)")
    return view_df, dropped


# ── Prepare all views ──
print("=" * 60)
print("Preparing feature views...")
print("=" * 60)

prepared_views = {}

for view_name, (source_df, feature_cols) in views.items():
    # For node-level views, aggregate per-ASN across snapshots
    if 'node' in view_name and 'asn' in source_df.columns and MODE == 'graph':
        # Filter ASNs with sufficient coverage
        snapshot_counts = source_df.groupby('asn')['snapshot_id'].nunique()
        n_snapshots = source_df['snapshot_id'].nunique()
        min_appearances = max(1, int(n_snapshots * MIN_NODE_SNAPSHOT_COVERAGE))
        good_asns = snapshot_counts[snapshot_counts >= min_appearances].index
        filtered = source_df[source_df['asn'].isin(good_asns)]
        print(f"\n  [{view_name}] ASN coverage filter: {len(good_asns)}/{len(snapshot_counts)} "
              f"ASes with >= {min_appearances}/{n_snapshots} snapshots")
        view_df, dropped = prepare_view(filtered, feature_cols, view_name)
    else:
        view_df, dropped = prepare_view(source_df, feature_cols, view_name)

    if view_df.shape[1] > 0:
        prepared_views[view_name] = view_df
    else:
        print(f"  [{view_name}] SKIPPED — no features remaining after cleanup")

print(f"\n{'='*60}")
print(f"Views ready for analysis: {len(prepared_views)}")
for name, df in prepared_views.items():
    print(f"  {name}: {df.shape}")

VIEWS = prepared_views

---
## 3. Distance Correlation (dCor)

Distance correlation (Székely et al., 2007) captures **both linear and non-linear** dependencies.
- dCor = 0 iff the variables are **independent** (stronger than Pearson = 0)
- dCor ∈ [0, 1]
- Particularly useful for detecting non-linear redundancies that Pearson/Spearman miss

In [ ]:

def compute_distance_correlation_matrix(df, view_name, max_samples=2000, verbose=True):
    """
    Compute pairwise distance correlation matrix.
    This is O(n^2 * p^2) so can be slow for large datasets.
    For large views, we optionally subsample.
    """
    if max_samples is not None and len(df) > max_samples:
        if verbose:
            print(f"  [{view_name}] Subsampling {max_samples}/{len(df)} rows for dCor")
        df_sub = df.sample(n=max_samples, random_state=42)
    else:
        df_sub = df

    if len(df_sub) < MIN_ROWS_FOR_DCOR:
        if verbose:
            print(f"  [{view_name}] Skipping dCor: need >= {MIN_ROWS_FOR_DCOR} rows, got {len(df_sub)}")
        return None

    cols = df_sub.columns.tolist()
    n_features = len(cols)
    dcor_matrix = np.ones((n_features, n_features))

    for i in range(n_features):
        for j in range(i + 1, n_features):
            x = df_sub[cols[i]].values
            y = df_sub[cols[j]].values
            try:
                dc = dcor.distance_correlation(x, y)
            except Exception:
                dc = 0.0
            dcor_matrix[i, j] = dc
            dcor_matrix[j, i] = dc

    dcor_df = pd.DataFrame(dcor_matrix, index=cols, columns=cols)
    return dcor_df


print("Computing Distance Correlation matrices...")
dcor_results = {}
for name, df in prepared_views.items():
    if df.shape[1] < 2:
        print(f"  [{name}] Skipping (< 2 features)")
        continue
    print(f"  [{name}] {df.shape[1]} features...")
    dcor_df = compute_distance_correlation_matrix(df, name)
    if dcor_df is None:
        continue
    dcor_results[name] = dcor_df
    print(f"    Done.")

print("\nDistance correlation computation complete.")


In [ ]:
# Visualize dCor matrices
for name, dcor_df in dcor_results.items():
    fig, ax = plt.subplots(figsize=(max(10, dcor_df.shape[1] * 0.6),
                                    max(8, dcor_df.shape[1] * 0.5)))
    mask = np.triu(np.ones_like(dcor_df, dtype=bool), k=1)
    sns.heatmap(dcor_df, mask=mask, annot=dcor_df.shape[1] <= 20,
                fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
                square=True, linewidths=0.5, ax=ax,
                annot_kws={'size': 8})
    ax.set_title(f'Distance Correlation — {name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'dcor_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Report highly correlated pairs
    high_dcor = []
    for i in range(dcor_df.shape[0]):
        for j in range(i + 1, dcor_df.shape[1]):
            if dcor_df.iloc[i, j] >= CORRELATION_THRESHOLD:
                high_dcor.append((
                    dcor_df.index[i], dcor_df.columns[j],
                    dcor_df.iloc[i, j]
                ))
    if high_dcor:
        print(f"\n[{name}] Highly distance-correlated pairs (dCor >= {CORRELATION_THRESHOLD}):")
        for f1, f2, val in sorted(high_dcor, key=lambda x: -x[2]):
            print(f"  {f1} <-> {f2}: dCor = {val:.4f}")
    else:
        print(f"\n[{name}] No pairs exceed dCor threshold {CORRELATION_THRESHOLD}")

---
## 4. Spearman & Pearson Correlation

- **Pearson**: measures linear correlation (assumes normality)
- **Spearman**: measures monotonic relationships (rank-based, robust to outliers)

Both are fast to compute and widely used as baseline redundancy detectors.

In [ ]:
def compute_correlations(df, view_name):
    """
    Compute Pearson and Spearman correlation matrices with p-values.
    """
    if len(df) < MIN_ROWS_FOR_CORRELATION:
        print(f"  [{view_name}] Skipping Pearson/Spearman: need >= {MIN_ROWS_FOR_CORRELATION} rows, got {len(df)}")
        return None

    cols = df.columns.tolist()
    n = len(cols)

    # Pearson
    pearson_r = df.corr(method='pearson')
    pearson_p = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)
    for i in range(n):
        for j in range(i + 1, n):
            r, p = sp_stats.pearsonr(df[cols[i]], df[cols[j]])
            pearson_p.iloc[i, j] = p
            pearson_p.iloc[j, i] = p

    # Spearman
    spearman_r = df.corr(method='spearman')
    spearman_p = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)
    for i in range(n):
        for j in range(i + 1, n):
            rho, p = sp_stats.spearmanr(df[cols[i]], df[cols[j]])
            spearman_p.iloc[i, j] = p
            spearman_p.iloc[j, i] = p

    return {
        'pearson_r': pearson_r, 'pearson_p': pearson_p,
        'spearman_r': spearman_r, 'spearman_p': spearman_p
    }


print("Computing Pearson & Spearman correlations...")
corr_results = {}
for name, df in prepared_views.items():
    if df.shape[1] < 2:
        print(f"  [{name}] Skipping (< 2 features)")
        continue
    print(f"  [{name}]...")
    corr_res = compute_correlations(df, name)
    if corr_res is None:
        continue
    corr_results[name] = corr_res

print("Done.")


In [ ]:
# Visualize Pearson & Spearman side by side
for name, res in corr_results.items():
    fig, axes = plt.subplots(1, 2, figsize=(
        max(20, res['pearson_r'].shape[1] * 1.0),
        max(8, res['pearson_r'].shape[1] * 0.5)))

    for ax, method, matrix in zip(axes,
                                   ['Pearson', 'Spearman'],
                                   [res['pearson_r'], res['spearman_r']]):
        mask = np.triu(np.ones_like(matrix, dtype=bool), k=1)
        sns.heatmap(matrix, mask=mask, annot=matrix.shape[1] <= 20,
                    fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                    square=True, linewidths=0.5, ax=ax,
                    annot_kws={'size': 7})
        ax.set_title(f'{method} — {name}', fontsize=13, fontweight='bold')

    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'pearson_spearman_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Report redundant pairs
    for method, r_matrix in [('Pearson', res['pearson_r']),
                              ('Spearman', res['spearman_r'])]:
        redundant = []
        for i in range(r_matrix.shape[0]):
            for j in range(i + 1, r_matrix.shape[1]):
                if abs(r_matrix.iloc[i, j]) >= CORRELATION_THRESHOLD:
                    redundant.append((
                        r_matrix.index[i], r_matrix.columns[j],
                        r_matrix.iloc[i, j]
                    ))
        if redundant:
            print(f"\n[{name}] {method} redundant pairs (|r| >= {CORRELATION_THRESHOLD}):")
            for f1, f2, val in sorted(redundant, key=lambda x: -abs(x[2])):
                print(f"  {f1} <-> {f2}: r = {val:.4f}")
        else:
            print(f"\n[{name}] {method}: no pairs exceed |r| >= {CORRELATION_THRESHOLD}")

---
## 5. Variance Inflation Factor (VIF) — Multicollinearity

VIF measures how much the variance of a regression coefficient is inflated due to collinearity.
- VIF = 1: no collinearity
- VIF = 5–10: moderate collinearity
- VIF > 10: severe multicollinearity (candidate for removal)

Reference: Belsley, Kuh & Welsch (1980). *Regression Diagnostics*.

In [ ]:

def compute_vif(df, view_name, max_samples=5000, verbose=True):
    """
    Compute VIF for each feature. Features are standardized first.
    For larger views, we optionally subsample for speed.
    """
    if max_samples is not None and len(df) > max_samples:
        if verbose:
            print(f"  [{view_name}] Subsampling {max_samples}/{len(df)} rows for VIF")
        df_sub = df.sample(n=max_samples, random_state=42)
    else:
        df_sub = df

    if len(df_sub) < MIN_ROWS_FOR_VIF:
        if verbose:
            print(f"  [{view_name}] Skipping VIF: need >= {MIN_ROWS_FOR_VIF} rows, got {len(df_sub)}")
        return pd.DataFrame({'Feature': df_sub.columns, 'VIF': np.nan})

    if df_sub.shape[1] < 2:
        return pd.DataFrame({'Feature': df_sub.columns, 'VIF': np.nan})

    if df_sub.shape[0] <= df_sub.shape[1]:
        if verbose:
            print(f"  [{view_name}] Skipping VIF: rows ({df_sub.shape[0]}) <= features ({df_sub.shape[1]})")
        return pd.DataFrame({'Feature': df_sub.columns, 'VIF': np.nan})

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_sub)
    X_df = pd.DataFrame(X_scaled, columns=df_sub.columns)

    vif_data = []
    for i, col in enumerate(X_df.columns):
        try:
            vif_val = variance_inflation_factor(X_df.values, i)
        except Exception:
            vif_val = np.inf
        vif_data.append({'Feature': col, 'VIF': vif_val})

    vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)
    return vif_df


print("Computing VIF for each view...")
print("=" * 60)
vif_results = {}
for name, df in prepared_views.items():
    if df.shape[1] < 2:
        print(f"  [{name}] Skipping (< 2 features)")
        continue
    print(f"\n[{name}]")
    vif_df = compute_vif(df, name)
    vif_results[name] = vif_df
    print(vif_df.to_string(index=False))

    multicollinear = vif_df[vif_df['VIF'] > VIF_THRESHOLD]
    if len(multicollinear) > 0:
        print(f"  >>> {len(multicollinear)} features with VIF > {VIF_THRESHOLD} "
              f"(multicollinear)")
    else:
        print(f"  >>> No features with VIF > {VIF_THRESHOLD}")


In [ ]:
# Visualize VIF as bar charts
for name, vif_df in vif_results.items():
    # Cap infinite VIF for visualization
    vif_plot = vif_df.copy()
    vif_plot['VIF'] = vif_plot['VIF'].replace([np.inf], vif_plot['VIF']
                                              [vif_plot['VIF'] != np.inf].max() * 1.5
                                              if (vif_plot['VIF'] != np.inf).any()
                                              else 100)

    fig, ax = plt.subplots(figsize=(max(10, len(vif_plot) * 0.5), 6))
    colors = ['#d32f2f' if v > VIF_THRESHOLD else '#1976d2'
              for v in vif_plot['VIF']]
    bars = ax.barh(vif_plot['Feature'], vif_plot['VIF'], color=colors)
    ax.axvline(x=VIF_THRESHOLD, color='red', linestyle='--', linewidth=1.5,
               label=f'VIF = {VIF_THRESHOLD}')
    ax.axvline(x=5, color='orange', linestyle='--', linewidth=1,
               label='VIF = 5 (moderate)')
    ax.set_xlabel('Variance Inflation Factor')
    ax.set_title(f'VIF — {name}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'vif_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

---
## 6. Hierarchical Clustering on Feature Correlations

Cluster features based on correlation distance `d = 1 - |corr|`.
Features within the same cluster (below a distance threshold) are near-duplicates;
we can keep one representative per cluster.

Reference: Hastie, Tibshirani & Friedman (2009). *Elements of Statistical Learning*.

In [ ]:
def hierarchical_cluster_features(df, view_name, threshold=None):
    """
    Perform hierarchical clustering on features using 1 - |Spearman| as distance.
    Returns cluster assignments and the linkage matrix.
    """
    if threshold is None:
        threshold = CLUSTER_DISTANCE_THRESHOLD

    corr = df.corr(method='spearman').abs()
    # Convert to distance: d = 1 - |r|
    dist_matrix = 1 - corr
    np.fill_diagonal(dist_matrix.values, 0)

    # Ensure symmetry and valid condensed form
    dist_matrix = (dist_matrix + dist_matrix.T) / 2
    condensed = squareform(dist_matrix.values, checks=False)

    # Ward-like linkage on correlation distance
    Z = linkage(condensed, method='average')
    clusters = fcluster(Z, t=threshold, criterion='distance')

    cluster_df = pd.DataFrame({
        'Feature': df.columns,
        'Cluster': clusters
    }).sort_values('Cluster')

    return Z, cluster_df, dist_matrix


print("Hierarchical Clustering of Features")
print("=" * 60)
cluster_results = {}

for name, df in prepared_views.items():
    if df.shape[1] < 3:
        print(f"  [{name}] Skipping (< 3 features)")
        continue

    Z, cluster_df, dist_matrix = hierarchical_cluster_features(df, name)
    cluster_results[name] = {'linkage': Z, 'clusters': cluster_df,
                              'dist_matrix': dist_matrix}

    print(f"\n[{name}]")
    n_clusters = cluster_df['Cluster'].nunique()
    print(f"  {df.shape[1]} features -> {n_clusters} clusters "
          f"(threshold = {CLUSTER_DISTANCE_THRESHOLD})")

    for cid in sorted(cluster_df['Cluster'].unique()):
        members = cluster_df[cluster_df['Cluster'] == cid]['Feature'].tolist()
        if len(members) > 1:
            print(f"  Cluster {cid} ({len(members)} features — REDUNDANT GROUP): {members}")
        else:
            print(f"  Cluster {cid}: {members[0]}")

In [ ]:
# Dendrograms
for name, res in cluster_results.items():
    Z = res['linkage']
    cluster_df = res['clusters']
    labels = prepared_views[name].columns.tolist()

    fig, ax = plt.subplots(figsize=(max(12, len(labels) * 0.6), 8))
    dn = dendrogram(Z, labels=labels, ax=ax, leaf_rotation=90,
                    color_threshold=CLUSTER_DISTANCE_THRESHOLD,
                    above_threshold_color='gray')
    ax.axhline(y=CLUSTER_DISTANCE_THRESHOLD, color='red', linestyle='--',
               linewidth=1.5, label=f'Threshold = {CLUSTER_DISTANCE_THRESHOLD}')
    ax.set_ylabel('Distance (1 - |Spearman|)')
    ax.set_title(f'Feature Dendrogram — {name}', fontsize=14, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'dendrogram_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

---
## 7. mRMR (Minimum Redundancy Maximum Relevance)

mRMR selects features that are **maximally relevant** to a target variable while being
**minimally redundant** among themselves.

Since this is an **unsupervised** intra-view analysis, we use a proxy target:
- For **graph-level**: use the first principal component (PC1) as a synthetic target
- For **node-level**: use degree centrality as the target (a natural proxy for node importance)

Reference: Peng, Long & Ding (2005). *IEEE Trans. PAMI*.

In [ ]:
from sklearn.decomposition import PCA


def run_mrmr(df, view_name, target_col=None, n_select=None):
    """
    Run mRMR feature selection.

    If target_col is None, uses PC1 as a synthetic target.
    Returns ranked list of selected features.
    """
    if not HAS_MRMR:
        print(f"  [{view_name}] mRMR not available. Skipping.")
        return None

    if n_select is None:
        n_select = max(3, df.shape[1] // 2)

    if target_col and target_col in df.columns:
        y = df[target_col]
        X = df.drop(columns=[target_col])
        print(f"  [{view_name}] Using '{target_col}' as target")
    else:
        # Use PC1 as synthetic target
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(df)
        pca = PCA(n_components=1, random_state=42)
        y = pd.Series(pca.fit_transform(X_scaled).ravel(),
                      index=df.index, name='PC1')
        X = df
        explained = pca.explained_variance_ratio_[0]
        print(f"  [{view_name}] Using PC1 as target "
              f"(explains {explained:.1%} variance)")

    n_select = min(n_select, X.shape[1])

    selected = mrmr_regression(
        X=X, y=y, K=n_select, relevance='f', redundancy='c',
        show_progress=False
    )
    return selected


print("mRMR Feature Selection")
print("=" * 60)
mrmr_results = {}
mrmr_targets = {}

for name, df in prepared_views.items():
    if df.shape[1] < 3:
        print(f"  [{name}] Skipping (< 3 features)")
        continue

    # Choose target based on view type
    if 'node' in name.lower():
        target = 'degree_centrality' if 'degree_centrality' in df.columns else None
    else:
        target = None  # use PC1

    mrmr_targets[name] = target
    selected = run_mrmr(df, name, target_col=target)
    mrmr_results[name] = selected

    if selected is not None and len(selected) > 0:
        selected_list = list(selected)
        print(f"  [{name}] mRMR ranking ({len(selected_list)} features):")
        for rank, feat in enumerate(selected_list, 1):
            print(f"    {rank}. {feat}")

        excluded_from_selection = {target} if target else set()
        eliminated = [f for f in df.columns if f not in selected_list and f not in excluded_from_selection]
        if eliminated:
            print(f"  Eliminated by mRMR: {eliminated}")
    print()


---
## 8. Mutual Information (MI)

Mutual information measures **general statistical dependence** (not just linear or monotonic).
Unlike dCor, MI is based on density estimation and can capture more complex dependencies.

Reference: Kraskov, Stögbauer & Grassberger (2004). *Physical Review E*.

In [ ]:

def compute_mutual_information_matrix(df, view_name, max_samples=3000, verbose=True):
    """
    Compute pairwise mutual information matrix using sklearn's KNN-based estimator.
    Normalized to [0, 1] using MI(X,Y) / sqrt(MI(X,X) * MI(Y,Y)).
    """
    if max_samples is not None and len(df) > max_samples:
        if verbose:
            print(f"  [{view_name}] Subsampling {max_samples}/{len(df)} rows for MI")
        df_sub = df.sample(n=max_samples, random_state=42)
    else:
        df_sub = df

    if len(df_sub) < MIN_ROWS_FOR_MI:
        if verbose:
            print(f"  [{view_name}] Skipping MI: need >= {MIN_ROWS_FOR_MI} rows, got {len(df_sub)}")
        return None

    cols = df_sub.columns.tolist()
    n = len(cols)
    n_neighbors = max(1, min(3, len(df_sub) - 1))

    self_mi = {}
    for col in cols:
        x = df_sub[col].values.reshape(-1, 1)
        try:
            mi = mutual_info_regression(
                x, df_sub[col].values,
                n_neighbors=n_neighbors, random_state=42
            )
            self_mi[col] = mi[0]
        except Exception:
            self_mi[col] = 0.0

    mi_matrix = np.zeros((n, n))
    np.fill_diagonal(mi_matrix, 1.0)

    for i in range(n):
        try:
            X_others = df_sub.drop(columns=[cols[i]]).values
            mi_scores = mutual_info_regression(
                X_others, df_sub[cols[i]].values,
                n_neighbors=n_neighbors, random_state=42
            )
        except Exception:
            mi_scores = np.zeros(n - 1)
        other_cols = [c for c in cols if c != cols[i]]
        for j_idx, other_col in enumerate(other_cols):
            j = cols.index(other_col)
            raw_mi = mi_scores[j_idx]
            denom = np.sqrt(self_mi[cols[i]] * self_mi[other_col])
            nmi = raw_mi / denom if denom > 0 else 0
            mi_matrix[i, j] = min(nmi, 1.0)

    mi_matrix = (mi_matrix + mi_matrix.T) / 2
    np.fill_diagonal(mi_matrix, 1.0)

    return pd.DataFrame(mi_matrix, index=cols, columns=cols)


print("Computing Mutual Information matrices...")
mi_results = {}
for name, df in prepared_views.items():
    if df.shape[1] < 2:
        print(f"  [{name}] Skipping (< 2 features)")
        continue
    print(f"  [{name}]...")
    mi_df = compute_mutual_information_matrix(df, name)
    if mi_df is None:
        continue
    mi_results[name] = mi_df
    print(f"    Done.")

print("\nMutual information computation complete.")


In [ ]:
# Visualize MI matrices
for name, mi_df in mi_results.items():
    fig, ax = plt.subplots(figsize=(max(10, mi_df.shape[1] * 0.6),
                                    max(8, mi_df.shape[1] * 0.5)))
    mask = np.triu(np.ones_like(mi_df, dtype=bool), k=1)
    sns.heatmap(mi_df, mask=mask, annot=mi_df.shape[1] <= 20,
                fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1,
                square=True, linewidths=0.5, ax=ax,
                annot_kws={'size': 8})
    ax.set_title(f'Normalized Mutual Information — {name}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'mi_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Report high-MI pairs
    high_mi = []
    for i in range(mi_df.shape[0]):
        for j in range(i + 1, mi_df.shape[1]):
            if mi_df.iloc[i, j] >= CORRELATION_THRESHOLD:
                high_mi.append((
                    mi_df.index[i], mi_df.columns[j], mi_df.iloc[i, j]
                ))
    if high_mi:
        print(f"\n[{name}] High NMI pairs (>= {CORRELATION_THRESHOLD}):")
        for f1, f2, val in sorted(high_mi, key=lambda x: -x[2]):
            print(f"  {f1} <-> {f2}: NMI = {val:.4f}")
    else:
        print(f"\n[{name}] No pairs exceed NMI threshold {CORRELATION_THRESHOLD}")

---
## 9. Condition Number Analysis

The condition number of the feature correlation matrix indicates overall numerical
stability and multicollinearity:
- CN < 30: well-conditioned
- CN 30–100: moderate collinearity
- CN > 100: severe collinearity
- CN > 1000: near-singular (extreme multicollinearity)

Reference: Belsley, Kuh & Welsch (1980). *Regression Diagnostics*.

In [ ]:

def compute_condition_summary(df, view_name, verbose=True):
    """
    Compute condition-number diagnostics for one cleaned feature view.
    Returns a summary dict, including the eigenvalue spectrum.
    """
    if df.shape[1] < 2:
        if verbose:
            print(f"\n[{view_name}]")
            print("  Skipping condition number: need >= 2 features")
        return {
            'view_name': view_name,
            'condition_number': np.nan,
            'status': 'SKIPPED',
            'max_eigenvalue': np.nan,
            'min_eigenvalue': np.nan,
            'near_zero_eigenvalues': np.nan,
            'eigenvalues': np.array([]),
        }

    if len(df) < MIN_ROWS_FOR_CONDITION:
        if verbose:
            print(f"\n[{view_name}]")
            print(f"  Skipping condition number: need >= {MIN_ROWS_FOR_CONDITION} rows, got {len(df)}")
        return {
            'view_name': view_name,
            'condition_number': np.nan,
            'status': 'SKIPPED',
            'max_eigenvalue': np.nan,
            'min_eigenvalue': np.nan,
            'near_zero_eigenvalues': np.nan,
            'eigenvalues': np.array([]),
        }

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df)
    corr_matrix = np.corrcoef(X_scaled.T)

    try:
        eigenvalues = np.linalg.eigvalsh(corr_matrix)
    except np.linalg.LinAlgError as e:
        if verbose:
            print(f"\n[{view_name}]")
            print(f"  Skipping condition number due to eigendecomposition failure: {e}")
        return {
            'view_name': view_name,
            'condition_number': np.nan,
            'status': 'FAILED',
            'max_eigenvalue': np.nan,
            'min_eigenvalue': np.nan,
            'near_zero_eigenvalues': np.nan,
            'eigenvalues': np.array([]),
        }

    eigenvalues = np.sort(eigenvalues)[::-1]
    min_eig = eigenvalues[eigenvalues > 1e-12].min() if (eigenvalues > 1e-12).any() else 1e-12
    condition_number = np.sqrt(eigenvalues[0] / min_eig)

    if condition_number < 30:
        status = "WELL-CONDITIONED"
    elif condition_number < 100:
        status = "MODERATE COLLINEARITY"
    elif condition_number < 1000:
        status = "SEVERE COLLINEARITY"
    else:
        status = "NEAR-SINGULAR (EXTREME)"

    if verbose:
        print(f"\n[{view_name}]")
        print(f"  Condition number: {condition_number:.1f} -> {status}")
        print(f"  Eigenvalue spectrum: max={eigenvalues[0]:.4f}, min={min_eig:.6f}")
        print(f"  Near-zero eigenvalues (< 0.01): {np.sum(eigenvalues < 0.01)}")

    return {
        'view_name': view_name,
        'condition_number': float(condition_number),
        'status': status,
        'max_eigenvalue': float(eigenvalues[0]),
        'min_eigenvalue': float(min_eig),
        'near_zero_eigenvalues': int(np.sum(eigenvalues < 0.01)),
        'eigenvalues': eigenvalues,
    }


print("Condition Number Analysis")
print("=" * 60)

condition_results = {}

for name, df in prepared_views.items():
    summary = compute_condition_summary(df, name, verbose=True)
    condition_results[name] = summary

    eigenvalues = summary['eigenvalues']
    if not isinstance(eigenvalues, np.ndarray) or len(eigenvalues) == 0 or np.isnan(summary['condition_number']):
        continue

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(1, len(eigenvalues) + 1), eigenvalues, color='steelblue')
    ax.axhline(y=0.01, color='red', linestyle='--', label='Near-zero threshold')
    ax.set_xlabel('Eigenvalue Index')
    ax.set_ylabel('Eigenvalue')
    ax.set_title(
        f'Correlation Matrix Eigenvalues — {name}\n'
        f'Condition Number = {summary["condition_number"]:.1f} ({summary["status"]})',
        fontsize=12,
        fontweight='bold'
    )
    ax.legend()
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'condition_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()


---
## 10. Consolidated Redundancy Report & Feature Selection

Combine evidence from all methods to make final recommendations about which features to keep/remove.

In [ ]:
def build_redundancy_report(view_name, df, corr_res, dcor_res, vif_res,
                             cluster_res, mrmr_res, mi_res, mrmr_target=None):
    """
    Build a consolidated per-feature redundancy score based on all methods.
    Higher score = more evidence of redundancy.
    """
    features = df.columns.tolist()
    report = pd.DataFrame({'Feature': features})

    # --- 1. Max |Pearson| with any other feature ---
    if corr_res and 'pearson_r' in corr_res:
        pearson = corr_res['pearson_r'].abs().copy()
        np.fill_diagonal(pearson.values, 0)
        report['Max_Pearson'] = [pearson.loc[f].max() for f in features]
    else:
        report['Max_Pearson'] = np.nan

    # --- 2. Max |Spearman| with any other feature ---
    if corr_res and 'spearman_r' in corr_res:
        spearman = corr_res['spearman_r'].abs().copy()
        np.fill_diagonal(spearman.values, 0)
        report['Max_Spearman'] = [spearman.loc[f].max() for f in features]
    else:
        report['Max_Spearman'] = np.nan

    # --- 3. Max dCor with any other feature ---
    if dcor_res is not None:
        dcor_abs = dcor_res.copy()
        np.fill_diagonal(dcor_abs.values, 0)
        report['Max_dCor'] = [dcor_abs.loc[f].max() for f in features]
    else:
        report['Max_dCor'] = np.nan

    # --- 4. Max NMI with any other feature ---
    if mi_res is not None:
        mi_abs = mi_res.copy()
        np.fill_diagonal(mi_abs.values, 0)
        report['Max_NMI'] = [mi_abs.loc[f].max() for f in features]
    else:
        report['Max_NMI'] = np.nan

    # --- 5. VIF ---
    if vif_res is not None:
        vif_map = dict(zip(vif_res['Feature'], vif_res['VIF']))
        report['VIF'] = report['Feature'].map(vif_map)
    else:
        report['VIF'] = np.nan

    # --- 6. Cluster size (features in multi-member clusters are redundant) ---
    if cluster_res:
        cluster_df = cluster_res['clusters']
        cluster_sizes = cluster_df.groupby('Cluster')['Feature'].transform('count')
        cluster_map = dict(zip(cluster_df['Feature'], cluster_sizes))
        report['Cluster_Size'] = report['Feature'].map(cluster_map).fillna(1)
    else:
        report['Cluster_Size'] = 1

    # --- 7. mRMR rank (not selected = penalized), except explicit target ---
    report['mRMR_Excluded_Target'] = report['Feature'].eq(mrmr_target) if mrmr_target else False

    if mrmr_res is not None and len(mrmr_res) > 0:
        selected_features = list(mrmr_res)
        mrmr_rank = {f: rank for rank, f in enumerate(selected_features, 1)}
        report['mRMR_Rank'] = report['Feature'].map(mrmr_rank)
        report['mRMR_Selected'] = report['Feature'].isin(selected_features)

        if mrmr_target:
            report.loc[report['Feature'] == mrmr_target, 'mRMR_Selected'] = True
            report.loc[report['Feature'] == mrmr_target, 'mRMR_Rank'] = 0
    else:
        report['mRMR_Rank'] = np.nan
        report['mRMR_Selected'] = np.nan

    # --- Composite redundancy score ---
    # Count how many methods flag each feature as redundant
    report['N_Flags'] = 0
    if 'Max_Pearson' in report:
        report['N_Flags'] += (report['Max_Pearson'] >= CORRELATION_THRESHOLD).astype(int)
    if 'Max_Spearman' in report:
        report['N_Flags'] += (report['Max_Spearman'] >= CORRELATION_THRESHOLD).astype(int)
    if 'Max_dCor' in report:
        report['N_Flags'] += (report['Max_dCor'] >= CORRELATION_THRESHOLD).astype(int)
    if 'Max_NMI' in report:
        report['N_Flags'] += (report['Max_NMI'] >= CORRELATION_THRESHOLD).astype(int)
    if 'VIF' in report:
        report['N_Flags'] += (report['VIF'] > VIF_THRESHOLD).astype(int)
    report['N_Flags'] += (report['Cluster_Size'] > 1).astype(int)

    if 'mRMR_Selected' in report and report['mRMR_Selected'].notna().any():
        mrmr_penalty = (~report['mRMR_Selected'].fillna(True)).astype(int)
        if 'mRMR_Excluded_Target' in report:
            mrmr_penalty = mrmr_penalty.where(~report['mRMR_Excluded_Target'], 0)
        report['N_Flags'] += mrmr_penalty

    report = report.sort_values('N_Flags', ascending=False)

    return report


print("\n" + "=" * 80)
print("CONSOLIDATED REDUNDANCY REPORT")
print("=" * 80)

redundancy_reports = {}
for name, df in prepared_views.items():
    if df.shape[1] < 2:
        continue

    report = build_redundancy_report(
        view_name=name,
        df=df,
        corr_res=corr_results.get(name),
        dcor_res=dcor_results.get(name),
        vif_res=vif_results.get(name),
        cluster_res=cluster_results.get(name),
        mrmr_res=mrmr_results.get(name),
        mi_res=mi_results.get(name),
        mrmr_target=mrmr_targets.get(name)
    )
    redundancy_reports[name] = report

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")

    # Format for display
    display_cols = ['Feature', 'Max_Pearson', 'Max_Spearman', 'Max_dCor',
                    'Max_NMI', 'VIF', 'Cluster_Size', 'mRMR_Rank',
                    'mRMR_Excluded_Target', 'N_Flags']
    display_cols = [c for c in display_cols if c in report.columns]
    print(report[display_cols].to_string(index=False, float_format='%.3f'))

    # Recommendations
    highly_redundant = report[report['N_Flags'] >= 4]['Feature'].tolist()
    moderately_redundant = report[
        (report['N_Flags'] >= 2) & (report['N_Flags'] < 4)
    ]['Feature'].tolist()
    clean = report[report['N_Flags'] < 2]['Feature'].tolist()

    print(f"\nRecommendations:")
    if highly_redundant:
        print(f"  REMOVE (flagged by 4+ methods): {highly_redundant}")
    if moderately_redundant:
        print(f"  REVIEW (flagged by 2-3 methods): {moderately_redundant}")
    print(f"  KEEP (flagged by 0-1 methods):   {clean}")


In [ ]:
# Visualize the consolidated report as a heatmap
for name, report in redundancy_reports.items():
    # Select numeric score columns
    score_cols = ['Max_Pearson', 'Max_Spearman', 'Max_dCor', 'Max_NMI']
    score_cols = [c for c in score_cols if c in report.columns
                  and report[c].notna().any()]

    if not score_cols:
        continue

    plot_df = report.set_index('Feature')[score_cols].copy()
    plot_df = plot_df.sort_index()

    fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.4)))
    sns.heatmap(plot_df, annot=True, fmt='.2f', cmap='YlOrRd',
                vmin=0, vmax=1, linewidths=0.5, ax=ax,
                annot_kws={'size': 9})
    ax.set_title(f'Max Pairwise Dependency per Feature — {name}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Method')
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'summary_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

---
## 11. Export Results

In [ ]:
# ============================================================================
# Export all results
# ============================================================================

# 1. Redundancy reports
for name, report in redundancy_reports.items():
    fname = f'redundancy_report_{name.replace(" ", "_").lower()}.csv'
    report.to_csv(ANALYSIS_DIR / fname, index=False)
    print(f"Saved: {ANALYSIS_DIR / fname}")

# 2. Correlation matrices
for name, res in corr_results.items():
    base = name.replace(" ", "_").lower()
    res['pearson_r'].to_csv(ANALYSIS_DIR / f'pearson_{base}.csv')
    res['spearman_r'].to_csv(ANALYSIS_DIR / f'spearman_{base}.csv')

# 3. Distance correlation matrices
for name, dcor_df in dcor_results.items():
    base = name.replace(" ", "_").lower()
    dcor_df.to_csv(ANALYSIS_DIR / f'dcor_{base}.csv')

# 4. MI matrices
for name, mi_df in mi_results.items():
    base = name.replace(" ", "_").lower()
    mi_df.to_csv(ANALYSIS_DIR / f'mi_{base}.csv')

# 5. VIF results
for name, vif_df in vif_results.items():
    base = name.replace(" ", "_").lower()
    vif_df.to_csv(ANALYSIS_DIR / f'vif_{base}.csv', index=False)

# 6. Cluster assignments
for name, res in cluster_results.items():
    base = name.replace(" ", "_").lower()
    res['clusters'].to_csv(ANALYSIS_DIR / f'clusters_{base}.csv', index=False)

# 7. mRMR rankings
for name, selected in mrmr_results.items():
    if selected is not None and len(selected) > 0:
        base = name.replace(" ", "_").lower()
        mrmr_df = pd.DataFrame({
            'Rank': range(1, len(selected) + 1),
            'Feature': list(selected)
        })
        mrmr_df.to_csv(ANALYSIS_DIR / f'mrmr_{base}.csv', index=False)

print(f"\nAll results exported to: {ANALYSIS_DIR}")
print("\nFiles:")
for f in sorted(ANALYSIS_DIR.iterdir()):
    if f.is_file():
        sz = f.stat().st_size / 1024
        print(f"  {f.name:<60} {sz:.1f} KB")


---
## 12. Final Feature Selection Summary

Based on all evidence, produce the final recommended feature set per view.

In [ ]:
print("=" * 80)
print("FINAL FEATURE SELECTION SUMMARY")
print("=" * 80)

final_selection = {}

for name, report in redundancy_reports.items():
    # Strategy: keep features flagged by <= 3 methods
    # From multi-member clusters, keep the feature with lowest N_Flags
    # (or highest mRMR rank if available)
    keep = []
    remove = []

    cluster_res = cluster_results.get(name)
    if cluster_res:
        cluster_df = cluster_res['clusters']
        for cid in cluster_df['Cluster'].unique():
            members = cluster_df[cluster_df['Cluster'] == cid]['Feature'].tolist()
            if len(members) == 1:
                keep.append(members[0])
            else:
                # Pick the member with fewest redundancy flags
                member_flags = report[report['Feature'].isin(members)].copy()
                # Prefer features with lower N_Flags; break ties with mRMR rank
                if 'mRMR_Rank' in member_flags.columns:
                    member_flags['sort_key'] = (
                        member_flags['N_Flags'] * 100 +
                        member_flags['mRMR_Rank'].fillna(999)
                    )
                else:
                    member_flags['sort_key'] = member_flags['N_Flags']
                member_flags = member_flags.sort_values('sort_key')
                best = member_flags.iloc[0]['Feature']
                keep.append(best)
                remove.extend([m for m in members if m != best])
    else:
        # No clustering — use N_Flags threshold
        keep = report[report['N_Flags'] < 4]['Feature'].tolist()
        remove = report[report['N_Flags'] >= 4]['Feature'].tolist()

    # Also remove features with extreme VIF that aren't cluster representatives
    vif_res = vif_results.get(name)
    if vif_res is not None:
        extreme_vif = vif_res[vif_res['VIF'] > 50]['Feature'].tolist()
        for f in extreme_vif:
            if f in keep and f not in remove:
                # Only remove if another feature in same cluster is kept
                if cluster_res:
                    cdf = cluster_res['clusters']
                    f_cluster = cdf[cdf['Feature'] == f]['Cluster'].values
                    if len(f_cluster) > 0:
                        same_cluster = cdf[cdf['Cluster'] == f_cluster[0]]['Feature'].tolist()
                        others_kept = [m for m in same_cluster if m in keep and m != f]
                        if others_kept:
                            keep.remove(f)
                            remove.append(f)

    final_selection[name] = {'keep': keep, 'remove': remove}

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(f"  Original features: {len(prepared_views[name].columns)}")
    print(f"  KEEP ({len(keep)}):   {keep}")
    print(f"  REMOVE ({len(remove)}): {remove}")
    print(f"  Reduction: {len(remove)}/{len(prepared_views[name].columns)} "
          f"({len(remove)/len(prepared_views[name].columns)*100:.0f}%) removed")

# Save final selection
import json
with open(ANALYSIS_DIR / 'final_feature_selection.json', 'w') as f:
    json.dump(final_selection, f, indent=2)
print(f"\nFinal selection saved to: {ANALYSIS_DIR / 'final_feature_selection.json'}")

In [ ]:

# ============================================================================
# Daily analysis suite (combined mode)
# ============================================================================

def classify_feature_domain(feature_name, graph_features, stat_features):
    if feature_name in graph_features:
        return 'graph'
    if feature_name in stat_features:
        return 'statistical'
    return 'other'


def classify_pair_domain(feature_a, feature_b, graph_features, stat_features):
    dom_a = classify_feature_domain(feature_a, graph_features, stat_features)
    dom_b = classify_feature_domain(feature_b, graph_features, stat_features)
    domains = tuple(sorted((dom_a, dom_b)))
    if domains == ('graph', 'graph'):
        return 'graph_graph'
    if domains == ('statistical', 'statistical'):
        return 'stat_stat'
    if domains == ('graph', 'statistical'):
        return 'graph_stat'
    return '_'.join(domains)


def prepare_daily_method_input(group_df, feature_cols):
    method_df = group_df[feature_cols].copy()
    method_df = method_df.apply(pd.to_numeric, errors='coerce')

    all_nan = method_df.columns[method_df.isna().all()].tolist()
    if all_nan:
        method_df = method_df.drop(columns=all_nan)

    constant = method_df.columns[method_df.nunique(dropna=True) <= 1].tolist()
    if constant:
        method_df = method_df.drop(columns=constant)

    if method_df.shape[1] == 0:
        return method_df, all_nan, constant

    method_df = method_df.fillna(method_df.median())
    return method_df, all_nan, constant


def reindex_square_matrix(matrix_df, full_feature_cols, diagonal_value=1.0):
    full_matrix = pd.DataFrame(0.0, index=full_feature_cols, columns=full_feature_cols)
    np.fill_diagonal(full_matrix.values, diagonal_value)
    if matrix_df is None or len(matrix_df) == 0:
        return full_matrix
    common = [c for c in matrix_df.index if c in full_feature_cols]
    full_matrix.loc[common, common] = matrix_df.loc[common, common]
    np.fill_diagonal(full_matrix.values, diagonal_value)
    return full_matrix


def build_daily_pairwise_matrix(method_df, full_feature_cols, method, day_label):
    nan_entries_filled = 0

    if method == 'pearson':
        raw_matrix = method_df.corr(method='pearson')
        nan_entries_filled = int(raw_matrix.isna().sum().sum())
        matrix = raw_matrix.fillna(0.0)
    elif method == 'spearman':
        raw_matrix = method_df.corr(method='spearman')
        nan_entries_filled = int(raw_matrix.isna().sum().sum())
        matrix = raw_matrix.fillna(0.0)
    elif method == 'dcor':
        matrix = compute_distance_correlation_matrix(
            method_df,
            f'daily_{method}_{day_label}',
            max_samples=DAILY_DCOR_MAX_SAMPLES,
            verbose=False,
        )
    elif method == 'mi':
        matrix = compute_mutual_information_matrix(
            method_df,
            f'daily_{method}_{day_label}',
            max_samples=DAILY_MI_MAX_SAMPLES,
            verbose=False,
        )
    else:
        raise ValueError(f"Unsupported daily pairwise method: {method}")

    return reindex_square_matrix(matrix, full_feature_cols, diagonal_value=1.0), nan_entries_filled


def compute_daily_pairwise_stability(base_df, prepared_df, feature_cols, graph_features, stat_features, method):
    daily_df = pd.concat(
        [base_df[['window_start']].reset_index(drop=True), prepared_df.reset_index(drop=True)],
        axis=1,
    )
    daily_df['window_date'] = daily_df['window_start'].dt.date

    daily_matrices = {}
    valid_days = []
    skipped_days = []
    day_summary_records = []

    for day, group in daily_df.groupby('window_date', sort=True):
        day_label = day.isoformat()
        if len(group) < DAILY_MIN_ROWS:
            skipped_days.append((day, len(group)))
            day_summary_records.append({
                'window_date': day,
                'method': method,
                'n_rows': int(len(group)),
                'n_features_used': 0,
                'n_features_dropped': int(len(feature_cols)),
                'nan_entries_filled': 0,
                'status': 'SKIPPED_MIN_ROWS',
            })
            continue

        method_df, all_nan, constant = prepare_daily_method_input(group, feature_cols)
        dropped = sorted(set(all_nan + constant))
        if method_df.shape[1] < 2:
            skipped_days.append((day, len(group)))
            day_summary_records.append({
                'window_date': day,
                'method': method,
                'n_rows': int(len(group)),
                'n_features_used': int(method_df.shape[1]),
                'n_features_dropped': int(len(dropped)),
                'nan_entries_filled': 0,
                'status': 'SKIPPED_TOO_FEW_FEATURES',
            })
            continue

        matrix_full, nan_entries_filled = build_daily_pairwise_matrix(method_df, feature_cols, method, day_label)
        daily_matrices[day] = matrix_full
        valid_days.append(day)
        day_summary_records.append({
            'window_date': day,
            'method': method,
            'n_rows': int(len(group)),
            'n_features_used': int(method_df.shape[1]),
            'n_features_dropped': int(len(dropped)),
            'nan_entries_filled': int(nan_entries_filled),
            'status': 'OK_FILLED_NAN' if nan_entries_filled > 0 else 'OK',
        })

    day_summary = pd.DataFrame(day_summary_records)

    pair_summary_columns = [
        'method', 'feature_a', 'feature_b', 'domain_a', 'domain_b', 'pair_domain',
        'n_days', 'mean_value', 'mean_abs_value', 'std_value', 'std_abs_value',
        'min_abs_value', 'max_abs_value', 'fraction_days_high_value', 'n_days_high_value',
        'sign_flip_count', 'mean_corr', 'mean_abs_corr', 'std_corr', 'std_abs_corr',
        'min_abs_corr', 'max_abs_corr', 'fraction_days_high_corr', 'n_days_high_corr',
        'stable_redundant'
    ]
    feature_summary_columns = [
        'method', 'Feature', 'Domain', 'best_partner', 'best_partner_pair_domain',
        'best_partner_mean_abs_value', 'best_partner_std_abs_value',
        'best_partner_fraction_days_high_value', 'best_partner_mean_abs_corr',
        'best_partner_std_abs_corr', 'best_partner_fraction_days_high_corr',
        'stable_pair_count', 'stable_graph_graph_pairs', 'stable_stat_stat_pairs',
        'stable_graph_stat_pairs', 'daily_mean_abs', 'daily_mean_std',
        'daily_mean_cv_raw', 'daily_mean_cv', 'cv_low_mean_flag'
    ]
    domain_summary_columns = [
        'pair_domain', 'stable_pair_count', 'mean_of_mean_abs_value',
        'mean_fraction_days_high_value', 'mean_std_abs_value'
    ]

    if not daily_matrices:
        return {
            'method': method,
            'pair_summary': pd.DataFrame(columns=pair_summary_columns),
            'feature_summary': pd.DataFrame(columns=feature_summary_columns),
            'domain_summary': pd.DataFrame(columns=domain_summary_columns),
            'daily_feature_stats': pd.DataFrame(columns=feature_cols),
            'daily_matrices': {},
            'day_summary': day_summary,
            'valid_days': [],
            'skipped_days': skipped_days,
        }

    ordered_days = list(valid_days)
    pair_records = []

    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            f1 = feature_cols[i]
            f2 = feature_cols[j]
            values = np.array([daily_matrices[day].loc[f1, f2] for day in ordered_days], dtype=float)
            abs_values = np.abs(values)
            high_days = abs_values >= STABLE_PAIR_ABS_CORR_THRESHOLD

            if method in {'pearson', 'spearman'}:
                nonzero_signs = np.sign(values[np.abs(values) > 1e-12])
                sign_flip_count = int(np.sum(nonzero_signs[1:] != nonzero_signs[:-1])) if len(nonzero_signs) > 1 else 0
            else:
                sign_flip_count = 0

            pair_records.append({
                'method': method,
                'feature_a': f1,
                'feature_b': f2,
                'domain_a': classify_feature_domain(f1, graph_features, stat_features),
                'domain_b': classify_feature_domain(f2, graph_features, stat_features),
                'pair_domain': classify_pair_domain(f1, f2, graph_features, stat_features),
                'n_days': len(values),
                'mean_value': float(values.mean()),
                'mean_abs_value': float(abs_values.mean()),
                'std_value': float(values.std(ddof=0)),
                'std_abs_value': float(abs_values.std(ddof=0)),
                'min_abs_value': float(abs_values.min()),
                'max_abs_value': float(abs_values.max()),
                'fraction_days_high_value': float(high_days.mean()),
                'n_days_high_value': int(high_days.sum()),
                'sign_flip_count': sign_flip_count,
                # compatibility columns used by later summary/label cells
                'mean_corr': float(values.mean()),
                'mean_abs_corr': float(abs_values.mean()),
                'std_corr': float(values.std(ddof=0)),
                'std_abs_corr': float(abs_values.std(ddof=0)),
                'min_abs_corr': float(abs_values.min()),
                'max_abs_corr': float(abs_values.max()),
                'fraction_days_high_corr': float(high_days.mean()),
                'n_days_high_corr': int(high_days.sum()),
            })

    pair_summary = pd.DataFrame(pair_records)
    pair_summary['stable_redundant'] = (
        (pair_summary['mean_abs_value'] >= STABLE_PAIR_ABS_CORR_THRESHOLD) &
        (pair_summary['fraction_days_high_value'] >= STABLE_PAIR_MIN_DAY_FRACTION) &
        (pair_summary['std_abs_value'] <= STABLE_PAIR_MAX_STD)
    )
    pair_summary = pair_summary.sort_values(
        ['stable_redundant', 'mean_abs_value', 'fraction_days_high_value', 'std_abs_value'],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

    daily_feature_stats = daily_df.groupby('window_date')[feature_cols].mean(numeric_only=True)
    daily_mean_abs = daily_feature_stats.abs().mean(axis=0)
    daily_mean_std = daily_feature_stats.std(axis=0, ddof=0)
    daily_mean_cv_raw = daily_mean_std / daily_mean_abs.replace(0, np.nan)
    cv_low_mean_flag = daily_mean_abs < DAILY_CV_MIN_MEAN_ABS
    daily_mean_cv = daily_mean_cv_raw.where(~cv_low_mean_flag, np.nan)

    feature_records = []
    for feature in feature_cols:
        related = pair_summary[
            (pair_summary['feature_a'] == feature) | (pair_summary['feature_b'] == feature)
        ].copy()
        related['partner'] = np.where(
            related['feature_a'] == feature,
            related['feature_b'],
            related['feature_a'],
        )
        related = related.sort_values(['mean_abs_value', 'fraction_days_high_value'], ascending=[False, False])
        top = related.iloc[0] if len(related) else None
        feature_records.append({
            'method': method,
            'Feature': feature,
            'Domain': classify_feature_domain(feature, graph_features, stat_features),
            'best_partner': None if top is None else top['partner'],
            'best_partner_pair_domain': None if top is None else top['pair_domain'],
            'best_partner_mean_abs_value': np.nan if top is None else top['mean_abs_value'],
            'best_partner_std_abs_value': np.nan if top is None else top['std_abs_value'],
            'best_partner_fraction_days_high_value': np.nan if top is None else top['fraction_days_high_value'],
            # compatibility columns used by label cell
            'best_partner_mean_abs_corr': np.nan if top is None else top['mean_abs_value'],
            'best_partner_std_abs_corr': np.nan if top is None else top['std_abs_value'],
            'best_partner_fraction_days_high_corr': np.nan if top is None else top['fraction_days_high_value'],
            'stable_pair_count': int(related['stable_redundant'].sum()),
            'stable_graph_graph_pairs': int(((related['pair_domain'] == 'graph_graph') & related['stable_redundant']).sum()),
            'stable_stat_stat_pairs': int(((related['pair_domain'] == 'stat_stat') & related['stable_redundant']).sum()),
            'stable_graph_stat_pairs': int(((related['pair_domain'] == 'graph_stat') & related['stable_redundant']).sum()),
            'daily_mean_abs': float(daily_mean_abs.get(feature, np.nan)),
            'daily_mean_std': float(daily_mean_std.get(feature, np.nan)),
            'daily_mean_cv_raw': float(daily_mean_cv_raw.get(feature, np.nan)),
            'daily_mean_cv': float(daily_mean_cv.get(feature, np.nan)),
            'cv_low_mean_flag': bool(cv_low_mean_flag.get(feature, False)),
        })

    feature_summary = pd.DataFrame(feature_records).sort_values(
        ['stable_pair_count', 'best_partner_mean_abs_value', 'daily_mean_cv'],
        ascending=[False, False, True],
    )

    stable_pairs = pair_summary[pair_summary['stable_redundant']].copy()
    if len(stable_pairs) > 0:
        domain_summary = stable_pairs.groupby('pair_domain').agg(
            stable_pair_count=('pair_domain', 'size'),
            mean_of_mean_abs_value=('mean_abs_value', 'mean'),
            mean_fraction_days_high_value=('fraction_days_high_value', 'mean'),
            mean_std_abs_value=('std_abs_value', 'mean'),
        ).reset_index().sort_values('stable_pair_count', ascending=False)
    else:
        domain_summary = pd.DataFrame(columns=[
            'pair_domain', 'stable_pair_count', 'mean_of_mean_abs_value',
            'mean_fraction_days_high_value', 'mean_std_abs_value'
        ])

    return {
        'method': method,
        'pair_summary': pair_summary,
        'feature_summary': feature_summary,
        'domain_summary': domain_summary,
        'daily_feature_stats': daily_feature_stats,
        'daily_matrices': daily_matrices,
        'day_summary': day_summary,
        'valid_days': valid_days,
        'skipped_days': skipped_days,
    }


def compute_daily_vif_suite(base_df, prepared_df, feature_cols):
    daily_df = pd.concat(
        [base_df[['window_start']].reset_index(drop=True), prepared_df.reset_index(drop=True)],
        axis=1,
    )
    daily_df['window_date'] = daily_df['window_start'].dt.date

    vif_records = []
    day_records = []

    for day, group in daily_df.groupby('window_date', sort=True):
        method_df, all_nan, constant = prepare_daily_method_input(group, feature_cols)
        dropped = sorted(set(all_nan + constant))

        if len(group) < DAILY_MIN_ROWS or method_df.shape[1] < 2:
            day_records.append({
                'window_date': day,
                'n_rows': int(len(group)),
                'n_features_used': int(method_df.shape[1]),
                'n_features_dropped': int(len(dropped)),
                'n_features_above_threshold': 0,
                'mean_vif': np.nan,
                'max_vif': np.nan,
                'status': 'SKIPPED',
            })
            for feature in feature_cols:
                vif_records.append({
                    'window_date': day,
                    'Feature': feature,
                    'VIF': np.nan,
                    'above_threshold': False,
                    'dropped_for_day': feature in dropped,
                })
            continue

        vif_df = compute_vif(method_df, f'daily_vif_{day.isoformat()}', max_samples=None, verbose=False)
        vif_map = dict(zip(vif_df['Feature'], vif_df['VIF']))

        valid_vifs = pd.Series(vif_map, dtype=float)
        day_records.append({
            'window_date': day,
            'n_rows': int(len(group)),
            'n_features_used': int(method_df.shape[1]),
            'n_features_dropped': int(len(dropped)),
            'n_features_above_threshold': int((valid_vifs > VIF_THRESHOLD).sum()) if len(valid_vifs) else 0,
            'mean_vif': float(valid_vifs.mean()) if len(valid_vifs) else np.nan,
            'max_vif': float(valid_vifs.max()) if len(valid_vifs) else np.nan,
            'status': 'OK',
        })

        for feature in feature_cols:
            vif_value = float(vif_map[feature]) if feature in vif_map else np.nan
            vif_records.append({
                'window_date': day,
                'Feature': feature,
                'VIF': vif_value,
                'above_threshold': bool(pd.notna(vif_value) and vif_value > VIF_THRESHOLD),
                'dropped_for_day': feature in dropped,
            })

    vif_by_day_feature = pd.DataFrame(vif_records)
    vif_day_summary = pd.DataFrame(day_records)
    vif_feature_summary = (
        vif_by_day_feature
        .groupby('Feature')
        .agg(
            mean_vif=('VIF', 'mean'),
            std_vif=('VIF', 'std'),
            max_vif=('VIF', 'max'),
            n_valid_days=('VIF', lambda s: int(s.notna().sum())),
            fraction_days_above_threshold=('above_threshold', 'mean'),
            fraction_days_dropped=('dropped_for_day', 'mean'),
        )
        .reset_index()
        .sort_values(['fraction_days_above_threshold', 'max_vif', 'mean_vif'], ascending=[False, False, False])
    )

    return {
        'by_day_feature': vif_by_day_feature,
        'day_summary': vif_day_summary,
        'feature_summary': vif_feature_summary,
    }


def compute_daily_condition_suite(base_df, prepared_df, feature_cols):
    daily_df = pd.concat(
        [base_df[['window_start']].reset_index(drop=True), prepared_df.reset_index(drop=True)],
        axis=1,
    )
    daily_df['window_date'] = daily_df['window_start'].dt.date

    condition_records = []

    for day, group in daily_df.groupby('window_date', sort=True):
        method_df, all_nan, constant = prepare_daily_method_input(group, feature_cols)
        dropped = sorted(set(all_nan + constant))
        summary = compute_condition_summary(method_df, f'daily_condition_{day.isoformat()}', verbose=False)
        condition_records.append({
            'window_date': day,
            'n_rows': int(len(group)),
            'n_features_used': int(method_df.shape[1]),
            'n_features_dropped': int(len(dropped)),
            'condition_number': summary['condition_number'],
            'status': summary['status'],
            'max_eigenvalue': summary['max_eigenvalue'],
            'min_eigenvalue': summary['min_eigenvalue'],
            'near_zero_eigenvalues': summary['near_zero_eigenvalues'],
        })

    condition_day_summary = pd.DataFrame(condition_records)
    condition_status_counts = (
        condition_day_summary['status']
        .value_counts()
        .rename_axis('status')
        .reset_index(name='count')
    )

    return {
        'day_summary': condition_day_summary,
        'status_counts': condition_status_counts,
    }


def build_daily_method_comparison(daily_pairwise_results):
    pair_domain_to_paper = {
        'graph_graph': 'graph_only',
        'stat_stat': 'stat_only',
        'graph_stat': 'cross_view',
    }

    method_overview_records = []
    domain_records = []
    pair_frames = []

    for method, result in daily_pairwise_results.items():
        stable_pairs = result['pair_summary']
        stable_pairs = stable_pairs[stable_pairs['stable_redundant']].copy()

        method_overview_records.append({
            'method': method,
            'valid_days': int(len(result['valid_days'])),
            'skipped_days': int(len(result['skipped_days'])),
            'stable_pair_count': int(len(stable_pairs)),
            'feature_count': int(len(result['feature_summary'])),
            'mean_abs_value': float(stable_pairs['mean_abs_value'].mean()) if len(stable_pairs) else np.nan,
            'mean_std_abs_value': float(stable_pairs['std_abs_value'].mean()) if len(stable_pairs) else np.nan,
        })

        if len(stable_pairs) == 0:
            continue

        stable_pairs['paper_domain'] = stable_pairs['pair_domain'].map(pair_domain_to_paper).fillna(stable_pairs['pair_domain'])

        domain_counts = stable_pairs.groupby('paper_domain').agg(
            stable_pair_count=('paper_domain', 'size'),
            mean_abs_value=('mean_abs_value', 'mean'),
            mean_fraction_days_high=('fraction_days_high_value', 'mean'),
            mean_std_abs_value=('std_abs_value', 'mean'),
        ).reset_index()
        domain_counts['method'] = method
        domain_records.append(domain_counts)

        pair_export = stable_pairs[[
            'feature_a', 'feature_b', 'pair_domain', 'paper_domain',
            'mean_abs_value', 'fraction_days_high_value', 'std_abs_value'
        ]].copy()
        pair_export['method'] = method
        pair_export['pair_key'] = pair_export.apply(
            lambda row: '||'.join(sorted([row['feature_a'], row['feature_b']])),
            axis=1,
        )
        pair_frames.append(pair_export)

    method_overview = pd.DataFrame(method_overview_records).sort_values('stable_pair_count', ascending=False)

    if domain_records:
        method_domain_summary = pd.concat(domain_records, ignore_index=True)
        method_domain_summary = method_domain_summary.sort_values(
            ['paper_domain', 'stable_pair_count', 'method'],
            ascending=[True, False, True],
        ).reset_index(drop=True)
    else:
        method_domain_summary = pd.DataFrame(columns=[
            'paper_domain', 'stable_pair_count', 'mean_abs_value',
            'mean_fraction_days_high', 'mean_std_abs_value', 'method'
        ])

    if pair_frames:
        all_pair_rows = pd.concat(pair_frames, ignore_index=True)
        pair_consensus = all_pair_rows.groupby(
            ['pair_key', 'feature_a', 'feature_b', 'pair_domain', 'paper_domain'],
            as_index=False,
        ).agg(
            n_methods=('method', 'nunique'),
            methods=('method', lambda s: ', '.join(sorted(set(s)))),
            mean_abs_value_mean=('mean_abs_value', 'mean'),
            mean_abs_value_min=('mean_abs_value', 'min'),
            mean_abs_value_max=('mean_abs_value', 'max'),
            mean_fraction_days_high=('fraction_days_high_value', 'mean'),
            mean_std_abs_value=('std_abs_value', 'mean'),
        ).sort_values(['n_methods', 'mean_abs_value_mean'], ascending=[False, False]).reset_index(drop=True)

        consensus_domain_summary = pair_consensus.groupby(['paper_domain', 'n_methods']).size().reset_index(name='pair_count')
        consensus_domain_summary = consensus_domain_summary.sort_values(
            ['n_methods', 'paper_domain'],
            ascending=[False, True],
        ).reset_index(drop=True)
    else:
        all_pair_rows = pd.DataFrame()
        pair_consensus = pd.DataFrame(columns=[
            'pair_key', 'feature_a', 'feature_b', 'pair_domain', 'paper_domain',
            'n_methods', 'methods', 'mean_abs_value_mean', 'mean_abs_value_min',
            'mean_abs_value_max', 'mean_fraction_days_high', 'mean_std_abs_value'
        ])
        consensus_domain_summary = pd.DataFrame(columns=['paper_domain', 'n_methods', 'pair_count'])

    return {
        'method_overview': method_overview,
        'method_domain_summary': method_domain_summary,
        'all_pair_rows': all_pair_rows,
        'pair_consensus': pair_consensus,
        'consensus_domain_summary': consensus_domain_summary,
    }



print("\n" + "=" * 80)
print("DAILY ANALYSIS SUITE")
print("=" * 80)

daily_pairwise_results = {}
daily_analysis_results = None
daily_vif_results = None
daily_condition_results = None
daily_method_comparison_results = None

if MODE != 'combined':
    print("Daily analysis is only supported for MODE='combined'.")
elif not ANALYZE_DAILY_STABILITY:
    print(f"Daily analysis disabled: {ANALYZE_DAILY_STABILITY_REASON}.")
else:
    source_key = f'{ANALYSIS_DATASET_LABEL}_cross_graph_vs_stat'
    if source_key not in prepared_views:
        available_sources = [k for k in prepared_views if k.endswith('_cross_graph_vs_stat')]
        raise KeyError(
            f"Requested ANALYSIS_DATASET_LABEL view '{source_key}' is unavailable. Available: {available_sources}"
        )

    source_df = unified_full_df
    prepared_df = prepared_views[source_key].copy()
    feature_cols = prepared_df.columns.tolist()
    graph_features = set([c for c in (FULL_GRAPH_LEVEL + FULL_NODE_LEVEL) if c in feature_cols])
    stat_features = set([c for c in FULL_STATISTICAL if c in feature_cols])

    print(f"Daily source view: {source_key}")
    print(f"  Rows: {len(prepared_df)}")
    print(f"  Features: {len(feature_cols)}")
    print(f"  Graph features: {len(graph_features)}")
    print(f"  Statistical features: {len(stat_features)}")
    print(
        f"  Stable pair threshold: |score| >= {STABLE_PAIR_ABS_CORR_THRESHOLD}, "
        f"fraction_days >= {STABLE_PAIR_MIN_DAY_FRACTION}, std_abs <= {STABLE_PAIR_MAX_STD}"
    )

    for method in DAILY_PAIRWISE_METHODS:
        print("\n" + "-" * 80)
        print(f"DAILY PAIRWISE METHOD: {method}")
        print("-" * 80)
        result = compute_daily_pairwise_stability(
            base_df=source_df,
            prepared_df=prepared_df,
            feature_cols=feature_cols,
            graph_features=graph_features,
            stat_features=stat_features,
            method=method,
        )
        daily_pairwise_results[method] = result

        print(f"  Valid days analyzed: {len(result['valid_days'])}")
        if result['skipped_days']:
            print(f"  Skipped days: {result['skipped_days']}")
        if len(result['valid_days']) == 0:
            print(
                f"  No daily analysis was possible for {method}. "
                f"This label subset is too sparse for DAILY_MIN_ROWS={DAILY_MIN_ROWS}."
            )
            print("  Try lowering DAILY_MIN_ROWS or disable the daily suite for anomaly-only subsets.")
            continue

        stable_pairs = result['pair_summary']
        stable_pairs = stable_pairs[stable_pairs['stable_redundant']]
        print("  Stable redundant pairs by domain:")
        if len(result['domain_summary']) > 0:
            print(result['domain_summary'].to_string(index=False, float_format='%.3f'))
        else:
            print("    None found under the current thresholds.")

    daily_analysis_results = daily_pairwise_results.get(DAILY_PRIMARY_METHOD)

    if DAILY_EXPORT_METHOD_COMPARISON:
        print("\n" + "-" * 80)
        print("DAILY METHOD COMPARISON")
        print("-" * 80)
        daily_method_comparison_results = build_daily_method_comparison(daily_pairwise_results)
        print(daily_method_comparison_results['method_overview'].to_string(index=False, float_format='%.3f'))

    if DAILY_ENABLE_VIF:
        print("\n" + "-" * 80)
        print("DAILY VIF SUMMARY")
        print("-" * 80)
        daily_vif_results = compute_daily_vif_suite(source_df, prepared_df, feature_cols)
        print(daily_vif_results['feature_summary'].head(20).to_string(index=False, float_format='%.3f'))

    if DAILY_ENABLE_CONDITION:
        print("\n" + "-" * 80)
        print("DAILY CONDITION NUMBER SUMMARY")
        print("-" * 80)
        daily_condition_results = compute_daily_condition_suite(source_df, prepared_df, feature_cols)
        print(daily_condition_results['day_summary'].to_string(index=False, float_format='%.3f'))


In [ ]:

# Visualize daily summaries for the primary daily method plus diagnostics.
def select_daily_heatmap_features(feature_summary, max_features, balance_domains=True):
    if len(feature_summary) == 0 or max_features <= 0:
        return []

    ranked = feature_summary.sort_values(
        ['stable_pair_count', 'best_partner_mean_abs_value', 'daily_mean_cv'],
        ascending=[False, False, True],
    ).copy()

    if not balance_domains:
        return ranked['Feature'].head(max_features).tolist()

    selected = []
    remaining_slots = max_features

    graph_quota = max_features // 2
    stat_quota = max_features - graph_quota

    for domain, quota in [('graph', graph_quota), ('statistical', stat_quota)]:
        domain_features = ranked[ranked['Domain'] == domain]['Feature'].tolist()
        take = domain_features[:quota]
        selected.extend(take)
        remaining_slots -= len(take)

    if remaining_slots > 0:
        remainder = ranked[~ranked['Feature'].isin(selected)]['Feature'].tolist()
        selected.extend(remainder[:remaining_slots])

    return selected[:max_features]


def save_daily_heatmap(matrix, method, day_label, output_path):
    feature_count = len(matrix.columns)
    fig_size = max(8, min(16, feature_count * 0.42))
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    if method in {'pearson', 'spearman'}:
        sns.heatmap(
            matrix,
            cmap='RdBu_r',
            vmin=-1,
            vmax=1,
            center=0,
            ax=ax,
            square=True,
            cbar_kws={'shrink': 0.8},
        )
    else:
        sns.heatmap(
            matrix,
            cmap='viridis',
            vmin=0,
            vmax=1,
            ax=ax,
            square=True,
            cbar_kws={'shrink': 0.8},
        )

    ax.set_title(f'{method.title()} Daily Heatmap: {day_label}', fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', labelrotation=90, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    plt.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches='tight')
    plt.close(fig)


daily_heatmap_feature_lists = {}
daily_heatmap_manifests = {}

if daily_analysis_results:
    primary_method = daily_analysis_results['method']
    stable_pairs = daily_analysis_results['pair_summary']
    stable_pairs = stable_pairs[stable_pairs['stable_redundant']].copy()
    feature_summary = daily_analysis_results['feature_summary'].copy()
    domain_summary = daily_analysis_results['domain_summary'].copy()

    if len(domain_summary) > 0:
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.barplot(data=domain_summary, x='pair_domain', y='stable_pair_count', ax=ax, palette='Set2')
        ax.set_title(f'Stable Daily Pairs by Domain ({primary_method.title()})', fontsize=13, fontweight='bold')
        ax.set_xlabel('Pair domain')
        ax.set_ylabel('Stable pair count')
        plt.tight_layout()
        fig.savefig(
            ANALYSIS_DIR / f'daily_stable_pair_counts_{primary_method}_{ANALYSIS_DATASET_LABEL}.png',
            dpi=150,
            bbox_inches='tight'
        )
        plt.show()

    if len(stable_pairs) > 0:
        top_plot = stable_pairs.head(min(TOP_DAILY_PAIR_COUNT, len(stable_pairs))).copy()
        top_plot['pair_label'] = top_plot['feature_a'] + ' <-> ' + top_plot['feature_b']
        fig, ax = plt.subplots(figsize=(12, max(6, len(top_plot) * 0.35)))
        sns.barplot(data=top_plot, y='pair_label', x='mean_abs_value', hue='pair_domain', dodge=False, ax=ax)
        ax.set_title(
            f'Top Stable Daily Pairs ({primary_method.title()})',
            fontsize=13,
            fontweight='bold'
        )
        ax.set_xlabel('Mean absolute daily score')
        ax.set_ylabel('Feature pair')
        plt.tight_layout()
        fig.savefig(
            ANALYSIS_DIR / f'daily_top_pairs_{primary_method}_{ANALYSIS_DATASET_LABEL}.png',
            dpi=150,
            bbox_inches='tight'
        )
        plt.show()

    if len(feature_summary) > 0:
        plot_features = feature_summary.head(min(TOP_DAILY_PAIR_COUNT, len(feature_summary))).copy()
        fig, ax = plt.subplots(figsize=(10, max(6, len(plot_features) * 0.3)))
        sns.barplot(data=plot_features, y='Feature', x='stable_pair_count', hue='Domain', dodge=False, ax=ax)
        ax.set_title(
            f'Features with the Most Stable Daily Partners ({primary_method.title()})',
            fontsize=13,
            fontweight='bold'
        )
        ax.set_xlabel('Number of stable redundant pairs')
        ax.set_ylabel('Feature')
        plt.tight_layout()
        fig.savefig(
            ANALYSIS_DIR / f'daily_feature_stability_{primary_method}_{ANALYSIS_DATASET_LABEL}.png',
            dpi=150,
            bbox_inches='tight'
        )
        plt.show()

if DAILY_EXPORT_HEATMAPS and daily_pairwise_results:
    print("\n" + "=" * 80)
    print("EXPORTING PER-DAY HEATMAPS")
    print("=" * 80)

    for method, result in daily_pairwise_results.items():
        if len(result['feature_summary']) == 0 or len(result['daily_matrices']) == 0:
            continue

        heatmap_features = select_daily_heatmap_features(
            result['feature_summary'],
            DAILY_HEATMAP_TOP_FEATURES,
            balance_domains=DAILY_HEATMAP_BALANCE_DOMAINS,
        )
        if not heatmap_features:
            continue

        heatmap_dir = ANALYSIS_DIR / f'daily_heatmaps_{method}_{ANALYSIS_DATASET_LABEL}'
        heatmap_dir.mkdir(parents=True, exist_ok=True)

        all_days = list(result['valid_days'])
        days_to_plot = all_days if DAILY_HEATMAP_MAX_DAYS is None else all_days[:DAILY_HEATMAP_MAX_DAYS]

        mean_matrix = None
        manifest_records = []

        for day in days_to_plot:
            day_label = day.isoformat()
            matrix = result['daily_matrices'][day].loc[heatmap_features, heatmap_features]
            heatmap_path = heatmap_dir / f'{method}_{day_label}_heatmap.png'
            save_daily_heatmap(matrix, method, day_label, heatmap_path)

            if mean_matrix is None:
                mean_matrix = matrix.copy()
            else:
                mean_matrix = mean_matrix.add(matrix, fill_value=0.0)

            manifest_records.append({
                'method': method,
                'artifact_kind': 'daily_heatmap',
                'window_date': day_label,
                'feature_count': len(heatmap_features),
                'file_name': heatmap_path.name,
                'file_path': str(heatmap_path),
            })

        if mean_matrix is not None and len(days_to_plot) > 0:
            mean_matrix = mean_matrix / len(days_to_plot)
            mean_path = heatmap_dir / f'{method}_mean_heatmap.png'
            save_daily_heatmap(mean_matrix, method, 'mean_across_days', mean_path)
            manifest_records.append({
                'method': method,
                'artifact_kind': 'mean_heatmap',
                'window_date': 'mean_across_days',
                'feature_count': len(heatmap_features),
                'file_name': mean_path.name,
                'file_path': str(mean_path),
            })

        daily_heatmap_feature_lists[method] = heatmap_features
        daily_heatmap_manifests[method] = pd.DataFrame(manifest_records)
        print(
            f"Saved {len(manifest_records)} heatmap artifacts for {method} "
            f"under {heatmap_dir}"
        )

if daily_method_comparison_results is not None and len(daily_method_comparison_results['method_overview']) > 0:
    overview_plot = daily_method_comparison_results['method_overview'].copy()
    domain_plot = daily_method_comparison_results['method_domain_summary'].copy()
    consensus_plot = daily_method_comparison_results['consensus_domain_summary'].copy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

    if len(domain_plot) > 0:
        sns.barplot(
            data=domain_plot,
            x='method',
            y='stable_pair_count',
            hue='paper_domain',
            palette='Set2',
            ax=axes[0],
        )
        axes[0].set_title('Stable Redundant Pairs by Method and Domain', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Method')
        axes[0].set_ylabel('Stable pair count')
        axes[0].legend(title='Domain')
    else:
        sns.barplot(data=overview_plot, x='method', y='stable_pair_count', color='steelblue', ax=axes[0])
        axes[0].set_title('Stable Redundant Pairs by Method', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Method')
        axes[0].set_ylabel('Stable pair count')

    if len(consensus_plot) > 0:
        sns.barplot(
            data=consensus_plot,
            x='n_methods',
            y='pair_count',
            hue='paper_domain',
            palette='Set1',
            ax=axes[1],
        )
        axes[1].set_title('Consensus Stable Pairs Across Methods', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Number of confirming methods')
        axes[1].set_ylabel('Pair count')
        axes[1].legend(title='Domain')
    else:
        sns.barplot(data=overview_plot, x='method', y='mean_abs_value', color='darkorange', ax=axes[1])
        axes[1].set_title('Average Stable-Pair Strength by Method', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Method')
        axes[1].set_ylabel('Mean absolute score')

    plt.tight_layout()
    fig.savefig(
        ANALYSIS_DIR / f'daily_method_comparison_compact_{ANALYSIS_DATASET_LABEL}.png',
        dpi=160,
        bbox_inches='tight'
    )
    plt.show()

if daily_vif_results is not None and len(daily_vif_results['feature_summary']) > 0:
    vif_plot = daily_vif_results['feature_summary'].head(min(TOP_DAILY_PAIR_COUNT, len(daily_vif_results['feature_summary']))).copy()
    fig, ax = plt.subplots(figsize=(10, max(6, len(vif_plot) * 0.3)))
    sns.barplot(data=vif_plot, y='Feature', x='fraction_days_above_threshold', color='teal', ax=ax)
    ax.set_title('Daily VIF Instability Across Days', fontsize=13, fontweight='bold')
    ax.set_xlabel(f'Fraction of days with VIF > {VIF_THRESHOLD}')
    ax.set_ylabel('Feature')
    plt.tight_layout()
    fig.savefig(
        ANALYSIS_DIR / f'daily_vif_instability_{ANALYSIS_DATASET_LABEL}.png',
        dpi=150,
        bbox_inches='tight'
    )
    plt.show()

if daily_condition_results is not None and len(daily_condition_results['day_summary']) > 0:
    cond_plot = daily_condition_results['day_summary'].copy()
    cond_plot['window_date_str'] = cond_plot['window_date'].astype(str)
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.lineplot(data=cond_plot, x='window_date_str', y='condition_number', marker='o', ax=ax)
    ax.set_title('Daily Condition Number Across Days', fontsize=13, fontweight='bold')
    ax.set_xlabel('Day')
    ax.set_ylabel('Condition number')
    ax.tick_params(axis='x', rotation=90)
    plt.tight_layout()
    fig.savefig(
        ANALYSIS_DIR / f'daily_condition_numbers_{ANALYSIS_DATASET_LABEL}.png',
        dpi=150,
        bbox_inches='tight'
    )
    plt.show()


In [ ]:

# Export daily analysis outputs.
if daily_pairwise_results:
    feature_daily_mean_written = False

    for method, result in daily_pairwise_results.items():
        method_suffix = f'{method}_{ANALYSIS_DATASET_LABEL}'
        pair_path = ANALYSIS_DIR / f'daily_pair_stability_{method_suffix}.csv'
        feature_path = ANALYSIS_DIR / f'daily_feature_stability_{method_suffix}.csv'
        domain_path = ANALYSIS_DIR / f'daily_domain_summary_{method_suffix}.csv'
        day_summary_path = ANALYSIS_DIR / f'daily_day_summary_{method_suffix}.csv'

        result['pair_summary'].to_csv(pair_path, index=False)
        result['feature_summary'].to_csv(feature_path, index=False)
        result['domain_summary'].to_csv(domain_path, index=False)
        result['day_summary'].to_csv(day_summary_path, index=False)

        print(f"Saved: {pair_path}")
        print(f"Saved: {feature_path}")
        print(f"Saved: {domain_path}")
        print(f"Saved: {day_summary_path}")

        if not feature_daily_mean_written:
            feature_daily_mean_path = ANALYSIS_DIR / f'daily_feature_means_{ANALYSIS_DATASET_LABEL}.csv'
            result['daily_feature_stats'].to_csv(feature_daily_mean_path)
            print(f"Saved: {feature_daily_mean_path}")
            feature_daily_mean_written = True

        if DAILY_EXPORT_PAIRWISE_MATRICES:
            matrices_dir = ANALYSIS_DIR / f'daily_matrices_{method_suffix}'
            matrices_dir.mkdir(parents=True, exist_ok=True)
            for day, matrix in result['daily_matrices'].items():
                matrix_path = matrices_dir / f'{method}_{day.isoformat()}.csv'
                matrix.to_csv(matrix_path)
            print(f"Saved daily matrices under: {matrices_dir}")

        if method in daily_heatmap_feature_lists:
            heatmap_features_path = ANALYSIS_DIR / f'daily_heatmap_features_{method_suffix}.csv'
            pd.DataFrame({'Feature': daily_heatmap_feature_lists[method]}).to_csv(heatmap_features_path, index=False)
            print(f"Saved: {heatmap_features_path}")

        if method in daily_heatmap_manifests:
            heatmap_manifest_path = ANALYSIS_DIR / f'daily_heatmap_manifest_{method_suffix}.csv'
            daily_heatmap_manifests[method].to_csv(heatmap_manifest_path, index=False)
            print(f"Saved: {heatmap_manifest_path}")

if daily_vif_results is not None:
    vif_values_path = ANALYSIS_DIR / f'daily_vif_values_{ANALYSIS_DATASET_LABEL}.csv'
    vif_feature_summary_path = ANALYSIS_DIR / f'daily_vif_feature_summary_{ANALYSIS_DATASET_LABEL}.csv'
    vif_day_summary_path = ANALYSIS_DIR / f'daily_vif_day_summary_{ANALYSIS_DATASET_LABEL}.csv'

    daily_vif_results['by_day_feature'].to_csv(vif_values_path, index=False)
    daily_vif_results['feature_summary'].to_csv(vif_feature_summary_path, index=False)
    daily_vif_results['day_summary'].to_csv(vif_day_summary_path, index=False)

    print(f"Saved: {vif_values_path}")
    print(f"Saved: {vif_feature_summary_path}")
    print(f"Saved: {vif_day_summary_path}")

if daily_condition_results is not None:
    condition_summary_path = ANALYSIS_DIR / f'daily_condition_summary_{ANALYSIS_DATASET_LABEL}.csv'
    condition_status_counts_path = ANALYSIS_DIR / f'daily_condition_status_counts_{ANALYSIS_DATASET_LABEL}.csv'

    daily_condition_results['day_summary'].to_csv(condition_summary_path, index=False)
    daily_condition_results['status_counts'].to_csv(condition_status_counts_path, index=False)

    print(f"Saved: {condition_summary_path}")
    print(f"Saved: {condition_status_counts_path}")


if daily_method_comparison_results is not None:
    method_overview_path = ANALYSIS_DIR / f'daily_method_overview_{ANALYSIS_DATASET_LABEL}.csv'
    method_domain_path = ANALYSIS_DIR / f'daily_method_domain_comparison_{ANALYSIS_DATASET_LABEL}.csv'
    pair_consensus_path = ANALYSIS_DIR / f'daily_pair_consensus_{ANALYSIS_DATASET_LABEL}.csv'
    consensus_domain_path = ANALYSIS_DIR / f'daily_pair_consensus_domain_summary_{ANALYSIS_DATASET_LABEL}.csv'

    daily_method_comparison_results['method_overview'].to_csv(method_overview_path, index=False)
    daily_method_comparison_results['method_domain_summary'].to_csv(method_domain_path, index=False)
    daily_method_comparison_results['pair_consensus'].to_csv(pair_consensus_path, index=False)
    daily_method_comparison_results['consensus_domain_summary'].to_csv(consensus_domain_path, index=False)

    print(f"Saved: {method_overview_path}")
    print(f"Saved: {method_domain_path}")
    print(f"Saved: {pair_consensus_path}")
    print(f"Saved: {consensus_domain_path}")



In [ ]:

# ============================================================================
# Feature interpretation labels for daily stability analysis
# ============================================================================

LABEL_STABLE_CV_THRESHOLD = 0.15
LABEL_DATE_SENSITIVE_CV_THRESHOLD = 0.35


def _top_partner_for_domain(stable_pairs, feature_name, pair_domain):
    subset = stable_pairs[
        ((stable_pairs['feature_a'] == feature_name) | (stable_pairs['feature_b'] == feature_name)) &
        (stable_pairs['pair_domain'] == pair_domain)
    ].copy()
    if len(subset) == 0:
        return None, np.nan
    subset['partner'] = np.where(subset['feature_a'] == feature_name, subset['feature_b'], subset['feature_a'])
    subset = subset.sort_values(['mean_abs_value', 'fraction_days_high_value'], ascending=[False, False])
    top = subset.iloc[0]
    return top['partner'], float(top['mean_abs_value'])


def build_feature_label_table(feature_summary, pair_summary):
    stable_pairs = pair_summary[pair_summary['stable_redundant']].copy()
    labeled = feature_summary.copy()

    temporal_labels = []
    redundancy_labels = []
    validation_labels = []
    primary_labels = []
    label_summaries = []
    top_cross_view_partner = []
    top_cross_view_corr = []
    top_within_view_partner = []
    top_within_view_corr = []

    for _, row in labeled.iterrows():
        cv = row['daily_mean_cv']
        if pd.isna(cv):
            temporal_label = 'insufficient_daily_info'
        elif cv <= LABEL_STABLE_CV_THRESHOLD:
            temporal_label = 'stable_within_as'
        elif cv >= LABEL_DATE_SENSITIVE_CV_THRESHOLD:
            temporal_label = 'date_sensitive'
        else:
            temporal_label = 'mixed_temporal_behavior'

        if row['stable_graph_stat_pairs'] > 0:
            redundancy_label = 'cross_view_redundant'
        elif (row['stable_graph_graph_pairs'] > 0) or (row['stable_stat_stat_pairs'] > 0):
            redundancy_label = 'within_view_redundant'
        else:
            redundancy_label = 'not_flagged_redundant'

        if temporal_label == 'stable_within_as':
            validation_label = 'needs_multi_as_validation'
        else:
            validation_label = 'single_as_evidence_only'

        if redundancy_label == 'cross_view_redundant':
            primary_label = 'cross_view_redundant'
        elif temporal_label == 'date_sensitive':
            primary_label = 'date_sensitive'
        elif temporal_label == 'stable_within_as':
            primary_label = 'stable_within_as'
        elif redundancy_label == 'within_view_redundant':
            primary_label = 'within_view_redundant'
        else:
            primary_label = 'mixed_temporal_behavior'

        cross_partner, cross_corr = _top_partner_for_domain(stable_pairs, row['Feature'], 'graph_stat')
        if row['Domain'] == 'graph':
            within_domain = 'graph_graph'
        elif row['Domain'] == 'statistical':
            within_domain = 'stat_stat'
        else:
            within_domain = None
        if within_domain is not None:
            within_partner, within_corr = _top_partner_for_domain(stable_pairs, row['Feature'], within_domain)
        else:
            within_partner, within_corr = (None, np.nan)

        summary_parts = [temporal_label]
        if redundancy_label != 'not_flagged_redundant':
            summary_parts.append(redundancy_label)
        if validation_label == 'needs_multi_as_validation':
            summary_parts.append(validation_label)

        temporal_labels.append(temporal_label)
        redundancy_labels.append(redundancy_label)
        validation_labels.append(validation_label)
        primary_labels.append(primary_label)
        label_summaries.append('; '.join(summary_parts))
        top_cross_view_partner.append(cross_partner)
        top_cross_view_corr.append(cross_corr)
        top_within_view_partner.append(within_partner)
        top_within_view_corr.append(within_corr)

    labeled['temporal_label'] = temporal_labels
    labeled['redundancy_label'] = redundancy_labels
    labeled['validation_label'] = validation_labels
    labeled['primary_label'] = primary_labels
    labeled['label_summary'] = label_summaries
    labeled['top_cross_view_partner'] = top_cross_view_partner
    labeled['top_cross_view_mean_abs_corr'] = top_cross_view_corr
    labeled['top_within_view_partner'] = top_within_view_partner
    labeled['top_within_view_mean_abs_corr'] = top_within_view_corr

    labeled = labeled.sort_values(
        ['primary_label', 'stable_pair_count', 'best_partner_mean_abs_value', 'daily_mean_cv'],
        ascending=[True, False, False, True],
    ).reset_index(drop=True)
    return labeled


def _temporal_priority(label):
    return {
        'stable_within_as': 3,
        'mixed_temporal_behavior': 2,
        'insufficient_daily_info': 1,
        'date_sensitive': 0,
    }.get(label, -1)


def _more_stable_feature(row):
    a_priority = _temporal_priority(row['feature_a_temporal_label'])
    b_priority = _temporal_priority(row['feature_b_temporal_label'])
    if a_priority > b_priority:
        return row['feature_a']
    if b_priority > a_priority:
        return row['feature_b']

    a_cv = row['feature_a_daily_mean_cv']
    b_cv = row['feature_b_daily_mean_cv']
    if pd.notna(a_cv) and pd.notna(b_cv):
        if a_cv < b_cv:
            return row['feature_a']
        if b_cv < a_cv:
            return row['feature_b']
    return None


def _paper_pair_note(row):
    if row['paper_domain'] == 'cross_view':
        base = 'Cross-view redundant pair'
    elif row['paper_domain'] == 'graph_only':
        base = 'Graph-only redundant pair'
    elif row['paper_domain'] == 'stat_only':
        base = 'Stat-only redundant pair'
    else:
        base = 'Stable redundant pair'

    preferred = row['candidate_more_stable_feature']
    if preferred is None:
        return base + '; both features show similar temporal stability'
    return base + f'; {preferred} is the temporally more stable feature'


def build_paper_redundancy_tables(pair_summary, feature_summary, label_table, method_name):
    stable_pairs = pair_summary[pair_summary['stable_redundant']].copy()
    stable_pairs['paper_domain'] = stable_pairs['pair_domain'].map({
        'graph_graph': 'graph_only',
        'stat_stat': 'stat_only',
        'graph_stat': 'cross_view',
    }).fillna(stable_pairs['pair_domain'])

    if len(stable_pairs) == 0:
        empty_pairs = pd.DataFrame()
        return {
            'method': method_name,
            'all_pairs': empty_pairs,
            'domain_summary': pd.DataFrame(),
            'feature_rollup': pd.DataFrame(),
            'split_tables': {
                'graph_only': pd.DataFrame(),
                'stat_only': pd.DataFrame(),
                'cross_view': pd.DataFrame(),
            },
        }

    feature_meta_cols = [
        'Feature', 'Domain', 'daily_mean_cv', 'stable_pair_count',
        'primary_label', 'temporal_label', 'redundancy_label', 'validation_label'
    ]
    feature_meta = label_table[feature_meta_cols].copy()

    feature_a_meta = feature_meta.rename(columns={
        'Feature': 'feature_a',
        'Domain': 'feature_a_domain',
        'daily_mean_cv': 'feature_a_daily_mean_cv',
        'stable_pair_count': 'feature_a_stable_pair_count',
        'primary_label': 'feature_a_primary_label',
        'temporal_label': 'feature_a_temporal_label',
        'redundancy_label': 'feature_a_redundancy_label',
        'validation_label': 'feature_a_validation_label',
    })
    feature_b_meta = feature_meta.rename(columns={
        'Feature': 'feature_b',
        'Domain': 'feature_b_domain',
        'daily_mean_cv': 'feature_b_daily_mean_cv',
        'stable_pair_count': 'feature_b_stable_pair_count',
        'primary_label': 'feature_b_primary_label',
        'temporal_label': 'feature_b_temporal_label',
        'redundancy_label': 'feature_b_redundancy_label',
        'validation_label': 'feature_b_validation_label',
    })

    paper_pairs = stable_pairs.merge(feature_a_meta, on='feature_a', how='left')
    paper_pairs = paper_pairs.merge(feature_b_meta, on='feature_b', how='left')

    paper_pairs['pair_label'] = paper_pairs['feature_a'] + ' <-> ' + paper_pairs['feature_b']
    paper_pairs['candidate_more_stable_feature'] = paper_pairs.apply(_more_stable_feature, axis=1)
    paper_pairs['paper_note'] = paper_pairs.apply(_paper_pair_note, axis=1)
    paper_pairs['paper_rank'] = (
        paper_pairs['mean_abs_value'] *
        paper_pairs['fraction_days_high_value'] *
        (1.0 - paper_pairs['std_abs_value'].clip(lower=0.0, upper=1.0))
    )

    paper_pairs = paper_pairs[[
        'paper_domain', 'pair_domain', 'pair_label', 'feature_a', 'feature_b',
        'mean_abs_value', 'fraction_days_high_value', 'std_abs_value', 'n_days',
        'feature_a_domain', 'feature_b_domain',
        'feature_a_temporal_label', 'feature_b_temporal_label',
        'feature_a_primary_label', 'feature_b_primary_label',
        'feature_a_daily_mean_cv', 'feature_b_daily_mean_cv',
        'feature_a_stable_pair_count', 'feature_b_stable_pair_count',
        'candidate_more_stable_feature', 'paper_note', 'paper_rank'
    ]].sort_values(
        ['paper_domain', 'paper_rank', 'mean_abs_value', 'fraction_days_high_value'],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)

    domain_summary = paper_pairs.groupby('paper_domain').agg(
        stable_pair_count=('pair_label', 'size'),
        mean_abs_value=('mean_abs_value', 'mean'),
        median_abs_value=('mean_abs_value', 'median'),
        mean_fraction_days_high=('fraction_days_high_value', 'mean'),
        mean_std_abs_value=('std_abs_value', 'mean'),
    ).reset_index().sort_values('stable_pair_count', ascending=False)

    mentions = pd.concat([
        paper_pairs[['paper_domain', 'feature_a']].rename(columns={'feature_a': 'Feature'}),
        paper_pairs[['paper_domain', 'feature_b']].rename(columns={'feature_b': 'Feature'}),
    ], ignore_index=True)

    feature_rollup = mentions.groupby(['Feature', 'paper_domain']).size().unstack(fill_value=0).reset_index()
    feature_rollup.columns.name = None
    for required_col in ['graph_only', 'stat_only', 'cross_view']:
        if required_col not in feature_rollup.columns:
            feature_rollup[required_col] = 0
    feature_rollup['stable_pair_mentions'] = (
        feature_rollup['graph_only'] + feature_rollup['stat_only'] + feature_rollup['cross_view']
    )
    feature_rollup = feature_rollup.merge(
        label_table[[
            'Feature', 'Domain', 'primary_label', 'temporal_label',
            'redundancy_label', 'daily_mean_cv', 'stable_pair_count'
        ]],
        on='Feature',
        how='left',
    ).sort_values(
        ['stable_pair_mentions', 'cross_view', 'graph_only', 'stat_only', 'daily_mean_cv'],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)

    split_tables = {
        'graph_only': paper_pairs[paper_pairs['paper_domain'] == 'graph_only'].copy(),
        'stat_only': paper_pairs[paper_pairs['paper_domain'] == 'stat_only'].copy(),
        'cross_view': paper_pairs[paper_pairs['paper_domain'] == 'cross_view'].copy(),
    }

    return {
        'method': method_name,
        'all_pairs': paper_pairs,
        'domain_summary': domain_summary,
        'feature_rollup': feature_rollup,
        'split_tables': split_tables,
    }


print("\n" + "=" * 80)
print("FEATURE INTERPRETATION LABELS")
print("=" * 80)

feature_label_results = {}
paper_redundancy_results = {}

if daily_analysis_results:
    print(f"Using primary daily method for labels: {daily_analysis_results['method']}")
    feature_label_table = build_feature_label_table(
        daily_analysis_results['feature_summary'],
        daily_analysis_results['pair_summary'],
    )
    feature_label_results[ANALYSIS_DATASET_LABEL] = feature_label_table

    print("Primary label counts:")
    print(feature_label_table['primary_label'].value_counts().to_string())

    print("\nTemporal label counts:")
    print(feature_label_table['temporal_label'].value_counts().to_string())

    print("\nRedundancy label counts:")
    print(feature_label_table['redundancy_label'].value_counts().to_string())

    print("\nTop labeled features:")
    print(feature_label_table[[
        'Feature', 'Domain', 'primary_label', 'temporal_label', 'redundancy_label',
        'stable_pair_count', 'daily_mean_cv', 'best_partner', 'top_cross_view_partner'
    ]].head(25).to_string(index=False, float_format='%.3f'))

    paper_redundancy_results[ANALYSIS_DATASET_LABEL] = build_paper_redundancy_tables(
        daily_analysis_results['pair_summary'],
        daily_analysis_results['feature_summary'],
        feature_label_table,
        daily_analysis_results['method'],
    )

    paper_summary = paper_redundancy_results[ANALYSIS_DATASET_LABEL]['domain_summary']
    print("\nPaper-ready redundancy domain summary:")
    if len(paper_summary) > 0:
        print(paper_summary.to_string(index=False, float_format='%.3f'))
    else:
        print("No stable redundant pairs found for the current daily method.")


In [ ]:

# Export feature interpretation labels and paper-ready redundancy tables
if feature_label_results:
    label_table = feature_label_results[ANALYSIS_DATASET_LABEL]
    label_path = ANALYSIS_DIR / f'daily_feature_labels_{DAILY_PRIMARY_METHOD}_{ANALYSIS_DATASET_LABEL}.csv'
    label_summary_path = ANALYSIS_DIR / f'daily_feature_label_summary_{DAILY_PRIMARY_METHOD}_{ANALYSIS_DATASET_LABEL}.csv'

    label_table.to_csv(label_path, index=False)

    label_summary = pd.DataFrame([
        {'group': 'primary_label', 'label': key, 'count': int(val)}
        for key, val in label_table['primary_label'].value_counts().items()
    ] + [
        {'group': 'temporal_label', 'label': key, 'count': int(val)}
        for key, val in label_table['temporal_label'].value_counts().items()
    ] + [
        {'group': 'redundancy_label', 'label': key, 'count': int(val)}
        for key, val in label_table['redundancy_label'].value_counts().items()
    ])
    label_summary.to_csv(label_summary_path, index=False)

    print(f"Saved: {label_path}")
    print(f"Saved: {label_summary_path}")

if paper_redundancy_results:
    paper_result = paper_redundancy_results[ANALYSIS_DATASET_LABEL]
    method_name = paper_result['method']

    paper_all_pairs_path = ANALYSIS_DIR / f'paper_redundant_pairs_{method_name}_{ANALYSIS_DATASET_LABEL}.csv'
    paper_domain_summary_path = ANALYSIS_DIR / f'paper_redundancy_domain_summary_{method_name}_{ANALYSIS_DATASET_LABEL}.csv'
    paper_feature_rollup_path = ANALYSIS_DIR / f'paper_redundancy_feature_rollup_{method_name}_{ANALYSIS_DATASET_LABEL}.csv'

    paper_result['all_pairs'].to_csv(paper_all_pairs_path, index=False)
    paper_result['domain_summary'].to_csv(paper_domain_summary_path, index=False)
    paper_result['feature_rollup'].to_csv(paper_feature_rollup_path, index=False)

    print(f"Saved: {paper_all_pairs_path}")
    print(f"Saved: {paper_domain_summary_path}")
    print(f"Saved: {paper_feature_rollup_path}")

    for domain_name, domain_df in paper_result['split_tables'].items():
        domain_path = ANALYSIS_DIR / (
            f'paper_redundant_pairs_{domain_name}_{method_name}_{ANALYSIS_DATASET_LABEL}.csv'
        )
        domain_df.to_csv(domain_path, index=False)
        print(f"Saved: {domain_path}")
